# Preliminary exploratory analysis: VR learning curves and trajectory stereotypy

Green Lab ViRMEn `wideLinearTrack` dataset. Four exploratory hypotheses:

1. **Learning** — (a) accuracy rises across sessions; (b) time-to-reward falls across sessions.
2. **Trajectory stereotypy** — (a) single-trial trajectories within a session become more similar
   across training; (b) session-average trajectories converge on the final trajectory.

**Task semantics.** `wideLinearTrack` has no choice point. From shaping stage 2 on the reward
tower is narrower than the maze, so an animal that hugs a wall completes the traverse and gets
**no reward**. That is the de facto "incorrect" trial, so:

> **accuracy := fraction of traverses that earned a reward**

Ground truth is always the `.mat` data. The tracking spreadsheet is hand-typed, contains no
formulas, and has arithmetically impossible rows — it is used only as a gross-error sanity check.

Standalone: the `mouse_imaging` package is **not importable** on this machine (`session.py`,
`behavior.py`, `tmaze.py` all `import functions as fc`, which does not exist under `Z:\Code\`).

In [1]:
import os, re, json, pickle, sys

# --- Windows DLL search path (must run BEFORE numpy is imported) ------------------------------
# A Jupyter kernel started by full interpreter path -- which is what VS Code does -- never runs
# conda's activate script, so this env's Library\bin is absent from PATH and the delay-loaded
# OpenBLAS that numpy links against cannot be found. The process then dies the first time
# anything touches BLAS, with Windows exit code 0xC06D007F (3228369023) and NO Python traceback:
# the kernel simply disappears, and restarting it does not help because every kernel is launched
# the same way. regenerate_figures.py documents the same failure mode for matplotlib.
#
# Verified by experiment: with these directories stripped from PATH, `a @ a` on a 400x400 array
# kills the interpreter; re-adding them makes BLAS, pandas groupby and matplotlib all succeed.
#
# The handles from add_dll_directory MUST be kept alive -- when one is garbage collected Windows
# removes the directory again -- hence the module-level list.
_DLL_DIR_HANDLES = []
if os.name == 'nt':
    for _d in (os.path.join(sys.prefix, 'Library', 'bin'),
               os.path.join(sys.prefix, 'Library', 'mingw-w64', 'bin'),
               os.path.join(sys.prefix, 'Library', 'usr', 'bin'),
               os.path.join(sys.prefix, 'DLLs')):
        if not os.path.isdir(_d):
            continue
        if _d not in os.environ.get('PATH', '').split(os.pathsep):
            os.environ['PATH'] = _d + os.pathsep + os.environ.get('PATH', '')
        try:
            _DLL_DIR_HANDLES.append(os.add_dll_directory(_d))
        except (AttributeError, OSError):
            pass

import numpy as np
import pandas as pd
import scipy.io
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

ROOT      = r"Y:\virmen"
CACHE_DIR = os.path.dirname(os.path.abspath("prelim_learning_analysis.ipynb"))
TRIAL_CSV = os.path.join(CACHE_DIR, "prelim_cache_trials.csv")
TRAJ_NPZ  = os.path.join(CACHE_DIR, "prelim_cache_traj.npz")

REBUILD = False          # True -> re-walk Y:\ (slow SMB share). False -> load cache.

ANIMALS = ['JB1', 'JB2', 'JB4', 'JB9']

# 0-based row indices into the 15 x nSamples `sessionData` matrix (verified empirically)
VIRMEN_COLUMNS = ['world_id', 'dx', 'dy', 'dh', 'x', 'y', 'h_int',
                  'inITI', 'reward', 'dt', 'lick', 'trial']

CM_PER_VU = 20.32 * np.pi / 86          # 0.7424 cm, per options.py:8
MIN_SAMPLES = 20000                     # ~6 min at ~56 Hz; drops aborted starts
N_BIN = 40                              # y-bins for trajectory resampling
MAX_DUR_S = 60.0                        # repo's own cap (behavior_analysis.py:77)
MAX_BACKTRACK_VU = 5.0                  # drops ~7% of trials, trajectory use only
REVERSAL_DEADBAND = 1.0                 # vu/s; |dy| below this is standing still, not motion
FS_BAND = (51.0, 61.0)                  # behavior-rig frame rate; 2P rig sits at ~39 Hz

TRAJ_MAZE = 'wideLinearTrack_200x20_friction05_floorStripe'

# Shaping ladder (labelling only -- geometry is measured from the data, never parsed from the name)
STAGE_ORDER = ['50x5', '50x10', '50x20', '100x20', '200x20']

def stage_of(exper_name):
    """Ladder label from experName. LABELLING ONLY -- see the geometry QC cell for why."""
    m = re.search(r'wideLinearTrack_(\d+)x(\d+)', exper_name or '')
    if not m:
        return None
    key = "%sx%s" % (m.group(1), m.group(2))
    return key if key in STAGE_ORDER else None

def stage_idx(exper_name):
    s = stage_of(exper_name)
    return STAGE_ORDER.index(s) if s in STAGE_ORDER else np.nan

## Plot style

Design follows the lab data-viz rules. Two choices worth stating, because they are deliberate:

* **No dual-axis charts.** Where two metrics are compared (dispersion vs correlation) they get
  stacked panels sharing an x-axis, never twin y-axes.
* **Per-animal panels are small multiples with a single series each**, so animal identity comes
  from the panel title rather than from a 4-colour categorical palette. Maze stage is an *ordered*
  variable (the difficulty ladder), so it gets a sequential single-hue ramp, not categorical hues.
  This sidesteps the categorical-palette CVD problem entirely rather than guessing at it.

In [2]:
INK        = '#0b0b0b'
INK_2      = '#52514e'
MUTED      = '#898781'
GRID       = '#e1e0d9'
BASELINE   = '#c3c2b7'
SURFACE    = '#fcfcfb'
BLUE       = '#2a78d6'
ORANGE     = '#eb6834'
GOOD       = '#0ca30c'
CRITICAL   = '#d03b3b'

# Ordinal ramp for the 5 shaping stages (light mode: start no lighter than step 250)
STAGE_COLORS = ['#86b6ef', '#5598e7', '#2a78d6', '#1c5cab', '#104281']
STAGE_FILL   = ['#eef5fe', '#e3eefd', '#d8e7fc', '#cde2fb', '#c2dbfa']

SEQ_CMAP = LinearSegmentedColormap.from_list(
    'labseq', ['#cde2fb', '#86b6ef', '#3987e5', '#256abf', '#184f95', '#0d366b'])
DIV_CMAP = LinearSegmentedColormap.from_list(
    'labdiv', ['#104281', '#2a78d6', '#9ec5f4', '#f0efec', '#f2b3b2', '#e34948', '#8f1f1e'])

mpl.rcParams.update({
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE,
    'axes.edgecolor': BASELINE, 'axes.labelcolor': INK_2, 'axes.titlecolor': INK,
    'axes.linewidth': 0.8, 'axes.grid': True, 'axes.axisbelow': True,
    'grid.color': GRID, 'grid.linewidth': 0.6,
    'xtick.color': MUTED, 'ytick.color': MUTED,
    'xtick.labelcolor': INK_2, 'ytick.labelcolor': INK_2,
    'text.color': INK, 'font.size': 9, 'axes.titlesize': 10, 'figure.dpi': 120,
    'lines.linewidth': 2.0, 'lines.markersize': 5,
    'legend.frameon': False, 'legend.fontsize': 8,
    'axes.spines.top': False, 'axes.spines.right': False,
})

# --- vector export -------------------------------------------------------------------
# 'none' keeps text as real text in the SVG instead of converting it to outlines, so labels
# stay editable (and re-typeable) in Illustrator / Inkscape.
mpl.rcParams['svg.fonttype'] = 'none'

FIG_DIR = os.path.join(CACHE_DIR, 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

def flatten_alpha(color, alpha, bg=SURFACE):
    """Opaque colour that looks identical to `color` drawn at `alpha` over `bg`.

    Using this instead of alpha= keeps `stroke-opacity` out of the SVG, so every line of a
    given kind carries a byte-identical style string and Illustrator's
    Select > Same > Appearance matches them all. Only use it for lines that do NOT overlap
    each other -- where overlap is the point (dense single-trial bundles) real alpha is needed.
    """
    import matplotlib.colors as mcolors
    c = np.array(mcolors.to_rgb(color)); b = np.array(mcolors.to_rgb(bg))
    return mcolors.to_hex(alpha * c + (1.0 - alpha) * b)

def illustrator_friendly(fig):
    """Drop per-artist clipping before vector export.

    Matplotlib clips every artist to the axes box and writes clip-path="url(#...)" onto each
    path. Illustrator imports each of those as its own <Clip Group>, and Select > Same >
    Appearance will not reach across that nesting -- which is why selecting one line fails to
    pick up its siblings. Nothing here actually extends past the axes, so removing the clip
    changes no pixels; it only flattens the group structure.
    """
    n = 0
    for ax in fig.get_axes():
        for artist in list(ax.lines) + list(ax.collections):
            artist.set_clip_on(False); n += 1
    return n

def save_fig(fig, name, formats=('svg', 'pdf', 'png')):
    """Write a figure to CACHE_DIR/figures/<name>.<ext>. Call BEFORE plt.show().

    SVG and PDF are both written: PDF usually imports into Illustrator with a flatter object
    tree than SVG, so if Select Same still misbehaves on the SVG, open the PDF instead.
    """
    illustrator_friendly(fig)
    out = []
    for ext in formats:
        p = os.path.join(FIG_DIR, '%s.%s' % (name, ext))
        fig.savefig(p, format=ext, dpi=300, bbox_inches='tight', facecolor=SURFACE)
        out.append(os.path.basename(p))
    print('saved %s: %s' % (name, ', '.join(out)))
    return out

def shade_stages(ax, stages):
    """Shade contiguous runs of a maze stage behind a per-session series.

    No inline text: early stages are only 1-2 sessions wide, so per-span labels
    collide into mush. Identity comes from the shared legend instead.
    """
    s = list(stages)
    if not s:
        return
    start = 0
    for i in range(1, len(s) + 1):
        if i == len(s) or s[i] != s[start]:
            k = s[start]
            if isinstance(k, str) and k in STAGE_ORDER:
                ax.axvspan(start - 0.5, i - 0.5, color=STAGE_FILL[STAGE_ORDER.index(k)],
                           zorder=0, lw=0)
            start = i

def stage_legend(fig, present):
    """One shared legend for the maze-stage shading, ordered along the ladder."""
    from matplotlib.patches import Patch
    h = [Patch(facecolor=STAGE_FILL[STAGE_ORDER.index(k)], edgecolor=BASELINE, lw=0.5, label=k)
         for k in STAGE_ORDER if k in present]
    if h:
        fig.legend(handles=h, title='maze stage', loc='lower center', ncol=len(h),
                   frameon=False, bbox_to_anchor=(0.5, 0.005))

def int_xaxis(ax):
    ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True))

def panel_title(ax, animal, rho, p, n):
    """Animal name and statistics on separate lines so they cannot collide."""
    ax.set_title(animal + chr(10) + fmt_rho(rho, p, n), loc='left', fontsize=9)

## Session manifest

Explicit and auditable: every session that exists on disk is listed, and excluded ones are
**commented out with the reason inline** so nothing is silently dropped and any exclusion can be
reversed by deleting one `#`.

Selection rule: **largest session per (animal, date, experName)**. `session_N` is sparse and
non-contiguous — `session_1` is frequently an aborted sub-2000-sample start — and keeping one
entry per (date, maze) preserves days where an animal ran two rungs of the ladder.

Three exclusion classes: **hardware** (reward-port failures, from the spreadsheet Notes),
**aborted starts** (< 20 000 samples, ~6 min), and **rig change** (JB9 from 260731).

In [3]:
SESSIONS = [
    # ---------------- JB1 ----------------
    ("JB1", "260527", "session_6"),              # 50x5_floorStripe
    # ("JB1", "260528", "session_1"),            # EXCLUDED (hardware): reward port tubing pinched, reward not released properly
    # ("JB1", "260601", "session_4"),            # EXCLUDED: aborted start, only 4510 samples
    ("JB1", "260601", "session_5"),              # 50x10_friction05_floorStripe
    ("JB1", "260601", "session_3"),              # 50x5_floorStripe
    # ("JB1", "260603", "session_3"),            # EXCLUDED (hardware): reward port drifted away mid-session; logged 15 pct is an artifact
    ("JB1", "260604", "session_2"),              # 100x20_friction05_floorStripe
    ("JB1", "260604", "session_1"),              # 50x20_friction05_floorStripe
    ("JB1", "260605", "session_1"),              # 100x20_friction05_floorStripe
    ("JB1", "260608", "session_1"),              # 100x20_friction05_floorStripe
    ("JB1", "260608", "session_2"),              # 200x20_friction05_floorStripe
    # ("JB1", "260609", "session_1"),            # EXCLUDED: aborted start, only 5686 samples
    ("JB1", "260609", "session_2"),              # 200x20_friction05_floorStripe
    # ("JB1", "260610", "session_1"),            # EXCLUDED: aborted start, only 14188 samples
    ("JB1", "260610", "session_2"),              # 200x20_friction05_floorStripe
    # ("JB1", "260612", "session_2"),            # EXCLUDED: aborted start, only 2898 samples
    ("JB1", "260612", "session_3"),              # 200x20_friction05_floorStripe
    ("JB1", "260615", "session_1"),              # 200x20_friction05_floorStripe
    ("JB1", "260616", "session_1"),              # 200x20_friction05_floorStripe
    ("JB1", "260617", "session_1"),              # 200x20_friction05_floorStripe
    ("JB1", "260618", "session_1"),              # 200x20_friction05_floorStripe
    ("JB1", "260622", "session_1"),              # 200x20_friction05_floorStripe
    ("JB1", "260629", "session_2"),              # 200x20_friction05_floorStripe
    ("JB1", "260707", "session_1"),              # 200x20_friction05_floorStripe
    ("JB1", "260714", "session_1"),              # 200x20_friction05_floorStripe
    ("JB1", "260715", "session_1"),              # 200x20_friction05_floorStripe
    ("JB1", "260717", "session_1"),              # 200x20_friction05_floorStripe
    ("JB1", "260720", "session_1"),              # 200x20_friction05_floorStripe
    ("JB1", "260721", "session_1"),              # 200x20_friction05_floorStripe
    ("JB1", "260727", "session_1"),              # 200x20_friction05_floorStripe
    ("JB1", "260730", "session_1"),              # 200x20_friction05_floorStripe
    ("JB1", "260803", "session_2"),              # 200x20_friction05_floorStripe
    ("JB1", "260805", "session_1"),              # 200x20_friction05_floorStripe
    ("JB1", "260807", "session_1"),              # 200x20_friction05_floorStripe
    ("JB1", "260810", "session_2"),              # 200x20_friction05_floorStripe
    # ---------------- JB2 ----------------
    ("JB2", "260527", "session_1"),              # 50x5_floorStripe
    ("JB2", "260528", "session_3"),              # 50x5_floorStripe
    ("JB2", "260601", "session_1"),              # 50x10_friction05_floorStripe
    # ("JB2", "260603", "session_1"),            # EXCLUDED (hardware): reward port drifted away over the session; logged 33 pct is an artifact
    ("JB2", "260604", "session_2"),              # 100x20_friction05_floorStripe
    # ("JB2", "260604", "session_1"),            # EXCLUDED: aborted start, only 3486 samples
    ("JB2", "260605", "session_1"),              # 100x20_friction05_floorStripe
    ("JB2", "260608", "session_1"),              # 100x20_friction05_floorStripe
    ("JB2", "260609", "session_1"),              # 100x20_friction05_floorStripe
    ("JB2", "260609", "session_2"),              # 200x20_friction05_floorStripe
    ("JB2", "260610", "session_1"),              # 100x20_friction05_floorStripe
    ("JB2", "260610", "session_2"),              # 200x20_friction05_floorStripe
    ("JB2", "260612", "session_2"),              # 100x20_friction05_floorStripe
    ("JB2", "260612", "session_3"),              # 200x20_friction05_floorStripe
    ("JB2", "260615", "session_1"),              # 200x20_friction05_floorStripe
    ("JB2", "260616", "session_1"),              # 200x20_friction05_floorStripe
    ("JB2", "260617", "session_1"),              # 200x20_friction05_floorStripe
    ("JB2", "260618", "session_3"),              # 200x20_friction05_floorStripe
    ("JB2", "260622", "session_1"),              # 200x20_friction05_floorStripe
    ("JB2", "260623", "session_1"),              # 200x20_friction05_floorStripe
    ("JB2", "260630", "session_1"),              # 200x20_friction05_floorStripe
    ("JB2", "260701", "session_1"),              # 200x20_friction05_floorStripe
    ("JB2", "260707", "session_1"),              # 200x20_friction05_floorStripe
    ("JB2", "260709", "session_1"),              # 200x20_friction05_floorStripe
    ("JB2", "260714", "session_1"),              # 200x20_friction05_floorStripe
    ("JB2", "260715", "session_1"),              # 200x20_friction05_floorStripe
    ("JB2", "260716", "session_1"),              # 200x20_friction05_floorStripe
    ("JB2", "260717", "session_1"),              # 200x20_friction05_floorStripe
    ("JB2", "260720", "session_1"),              # 200x20_friction05_floorStripe
    ("JB2", "260721", "session_1"),              # 200x20_friction05_floorStripe
    ("JB2", "260727", "session_1"),              # 200x20_friction05_floorStripe
    ("JB2", "260730", "session_1"),              # 200x20_friction05_floorStripe
    ("JB2", "260803", "session_1"),              # 200x20_friction05_floorStripe
    ("JB2", "260804", "session_1"),              # 200x20_friction05_floorStripe
    ("JB2", "260807", "session_1"),              # 200x20_friction05_floorStripe
    ("JB2", "260810", "session_1"),              # 200x20_friction05_floorStripe
    # ---------------- JB4 ----------------
    ("JB4", "260527", "session_1"),              # 50x5_floorStripe
    ("JB4", "260528", "session_1"),              # 50x5_floorStripe
    ("JB4", "260601", "session_1"),              # 50x10_friction05_floorStripe
    ("JB4", "260603", "session_1"),              # 50x20_friction05_floorStripe
    ("JB4", "260604", "session_1"),              # 100x20_friction05_floorStripe
    ("JB4", "260608", "session_1"),              # 100x20_friction05_floorStripe
    ("JB4", "260610", "session_1"),              # 100x20_friction05_floorStripe
    ("JB4", "260610", "session_2"),              # 200x20_friction05_floorStripe
    # ("JB4", "260612", "session_1"),            # EXCLUDED: aborted start, only 14732 samples
    ("JB4", "260612", "session_2"),              # 200x20_friction05_floorStripe
    ("JB4", "260615", "session_1"),              # 200x20_friction05_floorStripe
    ("JB4", "260616", "session_2"),              # 200x20_friction05_floorStripe
    ("JB4", "260617", "session_1"),              # 200x20_friction05_floorStripe
    ("JB4", "260618", "session_1"),              # 200x20_friction05_floorStripe
    ("JB4", "260622", "session_1"),              # 200x20_friction05_floorStripe
    ("JB4", "260626", "session_1"),              # 200x20_friction05_floorStripe
    ("JB4", "260629", "session_1"),              # 200x20_friction05_floorStripe
    ("JB4", "260707", "session_1"),              # 200x20_friction05_floorStripe
    ("JB4", "260709", "session_1"),              # 200x20_friction05_floorStripe
    ("JB4", "260710", "session_1"),              # 200x20_friction05_floorStripe
    ("JB4", "260714", "session_1"),              # 200x20_friction05_floorStripe
    # ("JB4", "260714", "session_3"),            # EXCLUDED: aborted start, only 997 samples
    ("JB4", "260715", "session_1"),              # 200x20_friction05_floorStripe
    ("JB4", "260716", "session_1"),              # 200x20_friction05_floorStripe
    ("JB4", "260717", "session_1"),              # 200x20_friction05_floorStripe
    ("JB4", "260720", "session_1"),              # 200x20_friction05_floorStripe
    ("JB4", "260721", "session_1"),              # 200x20_friction05_floorStripe
    ("JB4", "260723", "session_1"),              # 200x20_friction05_floorStripe
    ("JB4", "260727", "session_1"),              # 200x20_friction05_floorStripe
    ("JB4", "260731", "session_1"),              # 200x20_friction05_floorStripe
    ("JB4", "260803", "session_1"),              # 200x20_friction05_floorStripe
    ("JB4", "260804", "session_1"),              # 200x20_friction05_floorStripe
    ("JB4", "260807", "session_1"),              # 200x20_friction05_floorStripe
    ("JB4", "260810", "session_1"),              # 200x20_friction05_floorStripe
    # ---------------- JB9 ----------------
    # JB9 moved to the 2P rig on 260731 and never returned (fs 56 Hz -> 39 Hz).
    # Everything from 260731 on is excluded: different rig, ball suspension and
    # sampling rate. The notes only flagged 260731/260803 because behaviour visibly
    # collapsed there; 260804-260807 recovered but are still the 2P rig.
    ("JB9", "260701", "session_2"),              # 50x5_floorStripe
    # ("JB9", "260701", "session_1"),            # EXCLUDED: aborted start, only 556 samples
    ("JB9", "260702", "session_1"),              # 50x5_floorStripe
    ("JB9", "260707", "session_2"),              # 50x10_friction05_floorStripe
    # ("JB9", "260707", "session_1"),            # EXCLUDED: aborted start, only 1196 samples
    ("JB9", "260708", "session_1"),              # 50x10_friction05_floorStripe
    ("JB9", "260708", "session_2"),              # 50x20_friction05_floorStripe
    ("JB9", "260709", "session_1"),              # 50x20_friction05_floorStripe
    ("JB9", "260710", "session_1"),              # 50x20_friction05_floorStripe
    # ("JB9", "260714", "session_1"),            # EXCLUDED: aborted start, only 3212 samples
    ("JB9", "260714", "session_6"),              # 50x10_friction05_floorStripe
    ("JB9", "260714", "session_7"),              # 50x20_friction05_floorStripe
    ("JB9", "260715", "session_2"),              # 100x20_friction05_floorStripe
    ("JB9", "260715", "session_1"),              # 50x20_friction05_floorStripe
    ("JB9", "260716", "session_2"),              # 100x20_friction05_floorStripe
    ("JB9", "260716", "session_1"),              # 50x20_friction05_floorStripe
    ("JB9", "260717", "session_1"),              # 100x20_friction05_floorStripe
    ("JB9", "260720", "session_1"),              # 100x20_friction05_floorStripe
    ("JB9", "260720", "session_2"),              # 200x20_friction05_floorStripe
    ("JB9", "260721", "session_1"),              # 200x20_friction05_floorStripe
    # ("JB9", "260722", "session_1"),            # EXCLUDED (hardware): reward port pinched, not releasing water correctly
    ("JB9", "260727", "session_1"),              # 200x20_friction05_floorStripe
    # ("JB9", "260731", "session_3"),            # EXCLUDED (2P rig): fs=40 Hz
    # ("JB9", "260731", "session_2"),            # EXCLUDED (2P rig): fs=39 Hz
    # ("JB9", "260731", "session_1"),            # EXCLUDED (2P rig): fs=39 Hz
    # ("JB9", "260803", "session_2"),            # EXCLUDED (2P rig): fs=39 Hz
    # ("JB9", "260803", "session_3"),            # EXCLUDED (2P rig): fs=39 Hz
    # ("JB9", "260804", "session_1"),            # EXCLUDED (2P rig): fs=39 Hz
    # ("JB9", "260805", "session_1"),            # EXCLUDED (2P rig): fs=39 Hz
    # ("JB9", "260807", "session_1"),            # EXCLUDED (2P rig): fs=39 Hz
]
print("sessions in manifest: %d" % len(SESSIONS))
for a in ANIMALS:
    print("   %-5s %d" % (a, sum(1 for s in SESSIONS if s[0] == a)))

sessions in manifest: 113
   JB1   29
   JB2   34
   JB4   32
   JB9   18


## Loader

Adapted from `plot_trajectories_JB9.py:41-57`, which is the closest working template but is
missing three steps that matter:

* **`x *= -1`** (`wideLinearTrack.py:54`) — without it every plotted trajectory is mirrored
  left-right. The template omits this, so its published figures are flipped.
* **reward NaN -> 0 fill** (`session.py:583`) — `reward.max()` on an all-NaN trial returns NaN.
* **lick threshold `> 0.1`** (`session.py:579`).

Reward encoding varies between sessions (125/149 sparse-NaN, 23 mixed NaN+0, 1 dense 0/1) but
never contains anything other than 0, 1 or NaN, and rewards are **single-sample pulses** — so
`(reward == 1).sum()` is the reward count and no edge detection is needed.

In [4]:
def vr_columns(ncols):
    """User channels are inserted BEFORE the final 'trial' column (session.py:565-570)."""
    cols = list(VIRMEN_COLUMNS)
    if ncols > len(cols):
        extra = ['user%d' % i for i in range(ncols - len(cols))]
        cols = cols[:-1] + extra + cols[-1:]
    elif ncols < len(cols):
        raise ValueError("expected >= %d channels, got %d" % (len(cols), ncols))
    return cols

def session_path(animal, date, sess):
    return os.path.join(ROOT, animal, date, sess, 'sessionData.mat')

def load_vr(animal, date, sess):
    m = scipy.io.loadmat(session_path(animal, date, sess),
                         variable_names=['sessionData', 'experName', 'numRewards'],
                         squeeze_me=True)
    arr = np.asarray(m['sessionData']).T                  # (nSamples, nChannels)
    df = pd.DataFrame(arr, columns=vr_columns(arr.shape[1]))
    df['x'] = -df['x']                                    # wideLinearTrack.py:54
    df['reward'] = np.nan_to_num(df['reward'].to_numpy(), nan=0.0)
    df['lick'] = df['lick'] > 0.1                         # session.py:579
    df['t'] = df['dt'].cumsum()
    df['trial'] = np.nan_to_num(df['trial'].to_numpy()).astype(int)
    df['world_id'] = np.nan_to_num(df['world_id'].to_numpy()).astype(int)
    meta = dict(experName=str(m['experName']),
                numRewards=int(np.atleast_1d(m['numRewards'])[0]))
    return df, meta

## Build the per-trial table

One row per trial. Exclusion thresholds were set by measuring the data, not guessed:

* **No lower duration bound.** The shortest trial in the whole dataset is 0.91 s, so a 0.5 s floor
  removes nothing — and a 2 s floor would delete **35% of legitimate 50x20 trials**, which are
  genuinely fast because that track is only ~48 vu long. Duration thresholds must be stage-aware
  or absent; absent is simpler and safer.
* **`dur_s > 60` dropped** (1.03% of trials) as the disengagement filter — trials end at the tower,
  so a disengaged animal produces a *long* trial, never an incomplete one.
* **`max_backtrack > 5 vu`** for trajectory use only (~7%). Backtracking is common but tiny
  (median max excursion 1.81 vu); only the rare large reversal breaks binning by `y`.

Geometry (`track_len`) is **measured per session** as `y_max - median(y_start)`, never parsed from
`experName` — the name is not reliable (see the QC cell below).

In [5]:
def trial_table(animal, date, sess):
    df, meta = load_vr(animal, date, sess)
    y  = df['y'].to_numpy();  x  = df['x'].to_numpy()
    vx = df['dx'].to_numpy(); vy = df['dy'].to_numpy()   # VELOCITIES in vu/s (verified: diff(y) == dy*dt)
    dt = df['dt'].to_numpy(); tr = df['trial'].to_numpy()
    rew = (df['reward'].to_numpy() == 1)
    y_goal = float(y.max())                              # reward tower sits at the far end

    edges = np.flatnonzero(np.diff(tr) != 0) + 1
    starts = np.concatenate([[0], edges]); stops = np.concatenate([edges, [len(tr)]])

    recs = []
    for a, b in zip(starts, stops):
        if b - a < 3:
            continue
        yy = y[a:b]; xx = x[a:b]; dd = dt[a:b]
        ux = vx[a:b]; uy = vy[a:b]
        dur = float(dd.sum())
        run_max = np.maximum.accumulate(yy)

        # --- goal-directedness -------------------------------------------------
        # Approach rate toward G = (0, y_goal), computed ANALYTICALLY from the recorded
        # velocities. No np.gradient and no division by dt (sessions contain dt == 0
        # samples). Positive = moving toward the tower.
        tdist = np.hypot(xx, y_goal - yy)
        appr = ((y_goal - yy) * uy - xx * ux) / np.maximum(tdist, 1e-6)
        w = dd / dur if dur > 0 else np.full(len(dd), 1.0 / max(len(dd), 1))

        # Reversals of forward velocity, with a deadband so still-frame jitter is not counted
        sgn = np.sign(np.where(np.abs(uy) > REVERSAL_DEADBAND, uy, 0.0))
        sgn = sgn[sgn != 0]
        n_rev = int((np.diff(sgn) != 0).sum()) if len(sgn) > 1 else 0

        path = float((np.hypot(ux, uy) * dd).sum())
        net = float(np.hypot(xx[-1] - xx[0], yy[-1] - yy[0]))

        recs.append(dict(
            trial=int(tr[a]), n_samp=int(b - a),
            dur_s=dur,
            n_rewards=int(rew[a:b].sum()),
            y_start=float(yy[0]), y_end=float(yy[-1]), y_max=float(yy.max()),
            max_backtrack=float((run_max - yy).max()),
            x_mean=float(xx.mean()),
            frac_time_toward=float(w[appr > 0].sum()),
            frac_time_backward=float(w[uy < 0].sum()),
            n_reversals=n_rev,
            path_len=path,
            straightness=(net / path) if path > 0 else np.nan,
        ))
    t = pd.DataFrame(recs)
    if t.empty:
        return t

    sess_ymax = float(y.max())
    ystart_med = float(t['y_start'].median())
    t.insert(0, 'animal', animal); t.insert(1, 'date', date); t.insert(2, 'session', sess)
    t['experName'] = meta['experName']
    t['stage'] = stage_of(meta['experName'])
    t['stage_idx'] = stage_idx(meta['experName'])
    t['fs'] = 1.0 / float(np.median(dt))
    t['sess_y_max'] = sess_ymax
    t['track_len'] = sess_ymax - ystart_med           # measured, not from the name
    t['numRewards_mat'] = meta['numRewards']
    t['h_std'] = float(np.std(df['h_int'].to_numpy()))   # QC: must be 0 (view angle locked)
    t['rewarded'] = t['n_rewards'] > 0
    t['completed'] = t['y_max'] >= sess_ymax - 1.0    # QC only; measured at 99.8%
    t['dist_vu'] = t['y_end'] - t['y_start']
    t['speed_vu_s'] = t['dist_vu'] / t['dur_s']
    t['speed_cm_s'] = t['speed_vu_s'] * CM_PER_VU
    return t

def build_trials():
    out = []
    for i, (a, d, s) in enumerate(SESSIONS):
        try:
            t = trial_table(a, d, s)
            if not t.empty:
                out.append(t)
        except Exception as e:
            print("  FAILED %s %s %s: %s" % (a, d, s, e))
        if (i + 1) % 20 == 0:
            print("  ... %d/%d" % (i + 1, len(SESSIONS)))
    return pd.concat(out, ignore_index=True)

if REBUILD or not os.path.exists(TRIAL_CSV):
    print("rebuilding trial table from %s (slow) ..." % ROOT)
    trials = build_trials()
    trials.to_csv(TRIAL_CSV, index=False)
    print("wrote %s" % TRIAL_CSV)
else:
    trials = pd.read_csv(TRIAL_CSV)
    print("loaded cache %s" % TRIAL_CSV)

trials['keep'] = trials['dur_s'] <= MAX_DUR_S
print("trials: %d total, %d kept (%.2f%% dropped as dur > %gs)" % (
    len(trials), int(trials['keep'].sum()),
    100 * (1 - trials['keep'].mean()), MAX_DUR_S))
trials.head()

loaded cache c:\Users\JaclynBirnbaum\AppData\Local\Programs\Microsoft VS Code\prelim_cache_trials.csv
trials: 25126 total, 24865 kept (1.04% dropped as dur > 60s)


,animal,date,session,trial,n_samp,dur_s,n_rewards,y_start,y_end,y_max,...,sess_y_max,track_len,numRewards_mat,h_std,rewarded,completed,dist_vu,speed_vu_s,speed_cm_s,keep
0,JB1,260527,session_6,1,1703,28.858300,1,2.000000,54.990164,54.990164,...,54.998488,47.965243,9,0.0,True,True,52.990164,1.836219,1.363012,True
1,JB1,260527,session_6,2,4099,76.407408,1,7.095412,54.982414,54.982414,...,54.998488,47.965243,9,0.0,True,True,47.887002,0.626732,0.465219,False
2,JB1,260527,session_6,3,7889,147.907555,1,7.007556,54.944301,54.944301,...,54.998488,47.965243,9,0.0,True,True,47.936745,0.324099,0.240577,False
3,JB1,260527,session_6,4,5477,102.073510,1,7.002343,54.925883,54.925883,...,54.998488,47.965243,9,0.0,True,True,47.923540,0.469500,0.348507,False
4,JB1,260527,session_6,5,1438,26.941293,1,7.031038,54.935311,54.935311,...,54.998488,47.965243,9,0.0,True,True,47.904273,1.778099,1.319869,True


## QC — geometry, rig, reward counts

**`experName` does not reliably describe the maze.** A full cross-reference of name against
measured `y_max` across all 149 sessions on disk found one genuine mismatch:
`JB2/260528/session_2` is named `wideLinearTrack_50x5_floorStripe` but has `y_max = 15` — the
protocol's documented *"reduce distance between towers to 10 to get them started"* fallback, run
under an unchanged maze name. (It is already dropped here by the largest-session rule, but the
check stays so a future relabelled session is caught rather than silently pooled.)

The traverse distance also differs from the name by ~4%: all `50x*` mazes span 47.9 vu,
`100x20` spans 103.8, `200x20` spans 199.8.

In [6]:
sess = (trials.groupby(['animal', 'date', 'session'], as_index=False)
              .agg(experName=('experName', 'first'), stage=('stage', 'first'),
                   stage_idx=('stage_idx', 'first'), fs=('fs', 'first'),
                   y_max=('sess_y_max', 'first'), track_len=('track_len', 'first'),
                   y_start=('y_start', 'median'), n_trials=('trial', 'nunique'),
                   numRewards_mat=('numRewards_mat', 'first'),
                   n_rewards=('n_rewards', 'sum'), completed=('completed', 'mean')))

print("--- geometry: measured y_max vs the mode for each experName ---")
mode_ymax = sess.groupby('experName')['y_max'].agg(lambda s: s.round(1).mode().iloc[0])
sess['y_max_mode'] = sess['experName'].map(mode_ymax)
bad_geom = sess[(sess['y_max'] - sess['y_max_mode']).abs() > 0.5]
print(mode_ymax.to_string())
print("geometry mismatches in the analysis set: %d" % len(bad_geom))
if len(bad_geom):
    print(bad_geom[['animal', 'date', 'session', 'experName', 'y_max', 'y_max_mode']].to_string(index=False))

print("\n--- traverse distance: name vs measured ---")
print(sess.groupby('stage')['track_len'].agg(['count', 'median']).reindex(STAGE_ORDER).to_string())

print("\n--- rig: frame rate must sit in the behaviour-rig band %s ---" % (FS_BAND,))
bad_fs = sess[(sess['fs'] < FS_BAND[0]) | (sess['fs'] > FS_BAND[1])]
print("sessions outside the band: %d  (2P-rig sessions sit at ~39 Hz)" % len(bad_fs))
if len(bad_fs):
    print(bad_fs[['animal', 'date', 'session', 'fs']].to_string(index=False))

print("\n--- reward counts: (reward==1).sum() vs numRewards stored in the same .mat ---")
d = (sess['n_rewards'] - sess['numRewards_mat']).abs()
print("sessions differing by >2: %d / %d   (max |diff| = %d)" % (int((d > 2).sum()), len(sess), int(d.max())))

print("\n--- completion rate (expect ~99.8%: trials end AT the tower) ---")
print("mean completion = %.4f" % sess['completed'].mean())
assert len(bad_fs) == 0, "a session outside the frame-rate band reached the analysis set"
assert len(bad_geom) == 0, "a geometry mismatch reached the analysis set"
print("\nQC assertions passed.")

--- geometry: measured y_max vs the mode for each experName ---
experName
wideLinearTrack_100x20_friction05_floorStripe    105.0
wideLinearTrack_200x20_friction05_floorStripe    205.0
wideLinearTrack_50x10_friction05_floorStripe      55.0
wideLinearTrack_50x20_friction05_floorStripe      55.0
wideLinearTrack_50x5_floorStripe                  55.0
geometry mismatches in the analysis set: 0

--- traverse distance: name vs measured ---
        count      median
stage                    
50x5        8   47.899626
50x10       6   47.882052
50x20       8   47.838010
100x20     16  103.808024
200x20     75  199.817059

--- rig: frame rate must sit in the behaviour-rig band (51.0, 61.0) ---
sessions outside the band: 0  (2P-rig sessions sit at ~39 Hz)

--- reward counts: (reward==1).sum() vs numRewards stored in the same .mat ---
sessions differing by >2: 1 / 113   (max |diff| = 4)

--- completion rate (expect ~99.8%: trials end AT the tower) ---
mean completion = 0.9979

QC assertions passed.

## Build the trajectory cache

Restricted to `200x20`. Each trial's lateral position is resampled onto a **common `y` grid** —
not the arc-length parameterisation used by `plot_trajectories_JB9.py:76-86`. Arc length
confounds path length with track position, so "index k" is not the same place on the track in two
trials; the repo's own convention elsewhere (`aggregate_by_bin`, `wideLinearTrack.py:108`) is
y-binning, which is what we use.

Bins span `[y_start_med + 2, y_max - 2]`, derived per session from measured geometry, so the
~5.18 vu start offset and the tower are both excluded.

We also store **normalised-time speed profiles** `y(t)`: correlating those is immune to the
flat-line artefact that threatens `x(y)` correlations (see hypothesis 2a).

In [7]:
N_T = 50   # points in the normalised-time speed profile

def session_trajectories(animal, date, sess):
    df, meta = load_vr(animal, date, sess)
    if meta['experName'] != TRAJ_MAZE:
        return None
    y = df['y'].to_numpy(); x = df['x'].to_numpy()
    dt = df['dt'].to_numpy(); tr = df['trial'].to_numpy()
    rew = (df['reward'].to_numpy() == 1)

    edges = np.flatnonzero(np.diff(tr) != 0) + 1
    starts = np.concatenate([[0], edges]); stops = np.concatenate([edges, [len(tr)]])

    y_start_med = np.median([y[a] for a, b in zip(starts, stops) if b - a >= 3])
    lo, hi = y_start_med + 2.0, float(y.max()) - 2.0
    bin_edges = np.linspace(lo, hi, N_BIN + 1)
    ui = np.linspace(0.0, 1.0, N_T)

    X, Y, keep_tr, flag_rew, flag_bt = [], [], [], [], []
    for a, b in zip(starts, stops):
        if b - a < N_BIN:
            continue
        yy = y[a:b]; xx = x[a:b]; dd = dt[a:b]
        dur = dd.sum()
        if dur > MAX_DUR_S:                      # disengagement: still a hard exclusion
            continue
        # NOTE: reward failure and backtracking are recorded as FLAGS, not exclusions --
        # they are exactly the error trials hypothesis 3 is about. Hypotheses 2a/2b filter
        # on these flags at analysis time, so their results are unchanged.
        is_rew = bool(rew[a:b].any())
        backtrack = float((np.maximum.accumulate(yy) - yy).max())
        idx = np.digitize(yy, bin_edges) - 1
        ok = (idx >= 0) & (idx < N_BIN)
        if ok.sum() < N_BIN:
            continue
        cnt = np.bincount(idx[ok], minlength=N_BIN)
        if not (cnt > 0).all():
            continue
        X.append(np.bincount(idx[ok], weights=xx[ok], minlength=N_BIN) / cnt)
        tt = np.cumsum(dd); tt = (tt - tt[0]) / (tt[-1] - tt[0]) if tt[-1] > tt[0] else tt
        Y.append(np.interp(ui, tt, yy))
        keep_tr.append(int(tr[a])); flag_rew.append(is_rew); flag_bt.append(backtrack)
    if not X:
        return None
    return (np.asarray(X), np.asarray(Y), np.asarray(keep_tr),
            np.asarray(flag_rew, dtype=bool), np.asarray(flag_bt, dtype=float))

def build_traj():
    store = {}
    todo = [s for s in SESSIONS]
    for i, (a, d, s) in enumerate(todo):
        try:
            r = session_trajectories(a, d, s)
            if r is not None:
                X, Y, tr, frew, fbt = r
                key = "%s|%s|%s" % (a, d, s)
                store[key + "|X"] = X; store[key + "|Y"] = Y; store[key + "|tr"] = tr
                store[key + "|rew"] = frew; store[key + "|bt"] = fbt
        except Exception as e:
            print("  FAILED %s %s %s: %s" % (a, d, s, e))
        if (i + 1) % 20 == 0:
            print("  ... %d/%d" % (i + 1, len(todo)))
    return store

if REBUILD or not os.path.exists(TRAJ_NPZ):
    print("rebuilding trajectory cache (slow) ...")
    store = build_traj()
    np.savez_compressed(TRAJ_NPZ, **store)
    print("wrote %s" % TRAJ_NPZ)

_z = np.load(TRAJ_NPZ, allow_pickle=False)
TRAJ = {}
for k in _z.files:
    key, what = k.rsplit("|", 1)
    TRAJ.setdefault(key, {})[what] = _z[k]
# Hypotheses 2a/2b use the ORIGINAL trial set (rewarded, backtrack <= 5 vu) so their published
# values are reproduced exactly; hypothesis 3 uses every trial. Both views come from one cache.
def traj_clean(v):
    m = v['rew'] & (v['bt'] <= MAX_BACKTRACK_VU)
    return v['X'][m], v['Y'][m], v['tr'][m]

TRAJ = {k: v for k, v in TRAJ.items() if len(v.get('X', [])) >= 20}
TRAJ_C = {k: dict(zip(('X', 'Y', 'tr'), traj_clean(v))) for k, v in TRAJ.items()}
TRAJ_C = {k: v for k, v in TRAJ_C.items() if len(v['X']) >= 20}
print("sessions in cache: %d ; with >=20 clean trajectories: %d" % (len(TRAJ), len(TRAJ_C)))
for a in ANIMALS:
    print("   %-5s %d" % (a, sum(1 for k in TRAJ_C if k.startswith(a + "|"))))

sessions in cache: 75 ; with >=20 clean trajectories: 73
   JB1   22
   JB2   24
   JB4   24
   JB9   3


## Statistics

Sessions are autocorrelated day to day, so a nominal Spearman p-value is anticonservative. We use
a **circular-shift permutation null**, which preserves the autocorrelation of the metric series
while destroying its alignment to session order. The per-animal rho across ~22 sessions is the
real inferential unit — with only 3-4 animals there is no group-level test worth running
(a sign test on n=3 bottoms out at p=0.125 even with perfect agreement).

In [8]:
def _rank(a):
    a = np.asarray(a, float)
    order = a.argsort(); r = np.empty(len(a), float); r[order] = np.arange(len(a), dtype=float)
    vals, inv, cnt = np.unique(a, return_inverse=True, return_counts=True)
    for i in np.flatnonzero(cnt > 1):
        m = inv == i; r[m] = r[m].mean()
    return r

def _pearson(a, b):
    a = np.asarray(a, float) - np.mean(a); b = np.asarray(b, float) - np.mean(b)
    den = np.sqrt((a * a).sum() * (b * b).sum())
    return float((a * b).sum() / den) if den > 0 else np.nan

def spearman(x, y):
    return _pearson(_rank(x), _rank(y))

def spearman_circshift(y):
    """rho of y against session index, with a circular-shift permutation p-value."""
    y = np.asarray(y, float); n = len(y)
    i = np.arange(n, dtype=float)
    rho = spearman(i, y)
    if n < 5 or not np.isfinite(rho):
        return rho, np.nan, n
    null = [abs(spearman(i, np.roll(y, k))) for k in range(1, n)]
    p = (1.0 + sum(v >= abs(rho) for v in null)) / (1.0 + len(null))
    return rho, p, n

def fmt_rho(rho, p, n):
    ps = "n/a" if not np.isfinite(p) else ("%.3f" % p)
    return "rho=%+.3f  p=%s  (n=%d)" % (rho, ps, n)

RESULTS = []   # collected for the summary table
def record(hyp, metric, animal, rho, p, n, predicted):
    RESULTS.append(dict(hypothesis=hyp, metric=metric, animal=animal,
                        rho=rho, p=p, n=n, predicted=predicted,
                        supports=(np.sign(rho) == np.sign(predicted)) if np.isfinite(rho) else False))

## Per-session summary

In [ ]:
kept = trials[trials['keep']].copy()

def last_n_accuracy(g, n=50):
    g = g.sort_values('trial')
    return g['rewarded'].tail(n).mean()

ss = (kept.groupby(['animal', 'date', 'session'], as_index=False)
          .agg(experName=('experName', 'first'), stage=('stage', 'first'),
               stage_idx=('stage_idx', 'first'), track_len=('track_len', 'first'),
               n_trials=('trial', 'nunique'), accuracy=('rewarded', 'mean'),
               med_speed_cm_s=('speed_cm_s', 'median')))

ss['accuracy_last50'] = (kept.groupby(['animal', 'date', 'session'])
                              .apply(last_n_accuracy, include_groups=False).values)
rw = kept[kept['rewarded']]
ss['med_time_to_reward_s'] = (rw.groupby(['animal', 'date', 'session'])['dur_s']
                                .median().reindex(
                                    pd.MultiIndex.from_frame(ss[['animal', 'date', 'session']])).values)

ss = ss.sort_values(['animal', 'date', 'session']).reset_index(drop=True)
ss['sess_i'] = ss.groupby('animal').cumcount()
ss['stage_sess_i'] = ss.groupby(['animal', 'stage']).cumcount()
ss['cum_trials'] = ss.groupby('animal')['n_trials'].cumsum()
print(ss.groupby('animal')[['n_trials', 'accuracy', 'med_time_to_reward_s', 'med_speed_cm_s']]
        .agg(['count', 'median']).to_string())
ss.head()

       n_trials        accuracy           med_time_to_reward_s            med_speed_cm_s           
          count median    count    median                count     median          count     median
animal                                                                                             
JB1          29  208.0       29  0.926108                   29  10.699074             29  13.273142
JB2          33  227.0       33  0.686567                   33   9.853661             33  15.302725
JB4          32  194.5       32  0.745689                   32  10.201007             32  14.187869
JB9          18  215.0       18  0.739656                   18   5.441403             18  11.037364


,animal,date,session,experName,stage,stage_idx,track_len,n_trials,accuracy,med_speed_cm_s,accuracy_last50,med_time_to_reward_s,sess_i,stage_sess_i,cum_trials
0,JB1,260527,session_6,wideLinearTrack_50x5_floorStripe,50x5,0,47.965243,3,1.000000,1.363012,1.00,26.941293,0,0,3
1,JB1,260601,session_3,wideLinearTrack_50x5_floorStripe,50x5,0,47.896892,132,1.000000,9.102847,1.00,3.896312,1,1,135
2,JB1,260601,session_5,wideLinearTrack_50x10_friction05_floorStripe,50x10,1,47.863448,247,0.562753,8.288638,0.42,4.620005,2,0,382
3,JB1,260604,session_1,wideLinearTrack_50x20_friction05_floorStripe,50x20,2,47.821175,326,0.466258,16.502815,0.42,2.411555,3,0,708
4,JB1,260604,session_2,wideLinearTrack_100x20_friction05_floorStripe,100x20,3,103.805121,303,0.297030,17.127212,0.32,5.243510,4,0,1011


: 

## Hypothesis 1a — accuracy across sessions

Accuracy is the fraction of traverses that earned a reward.

**`50x5` is excluded from every panel below.** On that stage the reward tower spans the full width
of the maze, so an animal cannot complete a traverse without being rewarded: accuracy is 100% by
construction and carries no behavioural signal. The arc therefore starts at `50x10`.

Two views: **per animal** (small multiples, maze stage shaded behind the series) and **all animals
overlaid** on a single axis. The arc still mixes learning with the escalating difficulty of the
shaping ladder — see the within-stage trend statistics below and the limitations section.

In [ ]:
# 50x5 is omitted from hypothesis 1a: the reward tower spans the whole maze there, so a
# completed traverse is always rewarded and accuracy is 100% by construction.
ARC_STAGES = [s for s in STAGE_ORDER if s != '50x5']

arc = ss[ss['stage'].isin(ARC_STAGES)].copy()
arc = arc.sort_values(['animal', 'date', 'session']).reset_index(drop=True)
arc['sess_i'] = arc.groupby('animal').cumcount()          # re-indexed after dropping 50x5
print("sessions per animal, 50x5 dropped:")
print(arc.groupby('animal').size().to_string())

fig, axes = plt.subplots(2, 2, figsize=(11, 6.5), sharey=True)
for ax, a in zip(axes.ravel(), ANIMALS):
    g = arc[arc['animal'] == a]
    shade_stages(ax, list(g['stage']))
    ax.plot(g['sess_i'], 100 * g['accuracy'], color=BLUE, marker='o', zorder=3)
    rho, p, n = spearman_circshift(g['accuracy'].to_numpy())
    record('1a', 'accuracy', a, rho, p, n, +1)
    panel_title(ax, a, rho, p, n); int_xaxis(ax)
    ax.set_ylim(0, 105); ax.set_xlabel('session'); ax.set_ylabel('% rewarded traverses')
fig.suptitle('Hypothesis 1a — accuracy across the training arc, 50x5 excluded (shading = maze stage)',
             x=0.01, ha='left', fontsize=11, color=INK)
stage_legend(fig, set(arc['stage'].dropna()))
fig.tight_layout(rect=(0, 0.10, 1, 0.95))
plt.show()

In [ ]:
# All animals on one axis. Individual animals are deliberately NOT colour-coded here: they are all
# the same thin light line, and the across-animal mean carries the emphasis. No stage shading,
# because at a given session index the animals sit on different rungs of the ladder.
ANIMAL_COLORS  = {'JB1': BLUE, 'JB2': ORANGE, 'JB4': '#6a3d9a', 'JB9': INK}   # kept for other cells
ANIMAL_MARKERS = {'JB1': 'o',  'JB2': 's',    'JB4': '^',        'JB9': 'D'}

IND_COL  = '#9ec5f4'      # individual animals, de-emphasised
MEAN_LINE = '#104281'     # across-animal mean
SESS_XMAX = 20            # sessions shown on the overlay plots

def overlay_with_mean(ax, frame, value, animals, scale=1.0, x_max=SESS_XMAX,
                      col_ind=IND_COL, col_mean=MEAN_LINE):
    """Thin same-colour line per animal, plus a thick dark across-animal mean.

    Sessions are numbered from 1 for display (`sess_i` is 0-based internally) and the series are
    truncated at `x_max` BEFORE the mean is computed, so the mean and its legend describe exactly
    what is on screen rather than data that has been scrolled out of view.

    The mean is drawn SOLID only where every animal still contributes. Animals stop training at
    different points, so beyond that the mean would step purely because an animal dropped out,
    not because behaviour changed; that stretch is dashed and labelled separately.
    """
    series = {}
    for a in animals:
        g = frame[frame['animal'] == a].sort_values('sess_i')
        x = g['sess_i'].to_numpy() + 1                  # 1-based session numbering
        y = scale * g[value].to_numpy()
        if x_max is not None:
            keep = x <= x_max
            x, y = x[keep], y[keep]
        # Opaque flattened colour rather than alpha=0.85: identical on screen, but it keeps
        # stroke-opacity out of the SVG so every animal line shares one exact appearance.
        ax.plot(x, y, color=flatten_alpha(col_ind, 0.85), linewidth=0.9,
                zorder=2, label='_nolegend_')
        series[a] = pd.Series(y, index=x)

    M = pd.DataFrame(series).sort_index()
    n_present = M.notna().sum(axis=1)
    mean = M.mean(axis=1)
    full = (n_present == len(animals)).to_numpy()

    ax.plot(mean.index[full], mean.to_numpy()[full], color=col_mean, linewidth=3.0, zorder=4,
            label='across-animal mean (n=%d)' % len(animals))
    if (~full).any():
        part = mean.to_numpy().astype(float).copy()
        part[full] = np.nan
        cut = int(np.flatnonzero(~full)[0])
        if cut > 0:
            part[cut - 1] = mean.to_numpy()[cut - 1]      # bridge to the solid segment
        ax.plot(mean.index, part, color=col_mean, linewidth=2.0, linestyle='--', zorder=4,
                label='mean, fewer animals')
    ax.set_xlim(0.5, x_max + 0.5)
    ax.grid(False)
    return M

fig, ax = plt.subplots(figsize=(8.4, 4.4))
overlay_with_mean(ax, arc, 'accuracy', ANIMALS, scale=100.0)
ax.axhline(70, color=MUTED, linestyle='--', linewidth=1.2, zorder=1)
ax.text(0.995, 70, ' 70%', transform=ax.get_yaxis_transform(), ha='right', va='bottom',
        fontsize=8, color=MUTED)
ax.set_ylim(0, 105); int_xaxis(ax)
ax.set_xlabel('session (50x5 excluded)'); ax.set_ylabel('% rewarded traverses')
ax.legend(loc='lower right', frameon=False)
fig.suptitle('Hypothesis 1a - accuracy across the training arc, all animals overlaid',
             x=0.01, ha='left', fontsize=11, color=INK)
fig.tight_layout(rect=(0, 0, 1, 0.93))
save_fig(fig, 'h1a_accuracy_overlay')
plt.show()

In [ ]:
# Within-stage trends: statistics only, no panels. 50x5 is skipped for the same reason as above.
print("Hypothesis 1a — within-stage accuracy trends (stages with >=5 sessions):")
for a in ANIMALS:
    for st in ARC_STAGES:
        h = ss[(ss['animal'] == a) & (ss['stage'] == st)]
        if len(h) >= 5:
            rho, p, n = spearman_circshift(h['accuracy'].to_numpy())
            record('1a', 'accuracy (%s)' % st, a, rho, p, n, +1)
            print("   %-5s %-7s %s" % (a, st, fmt_rho(rho, p, n)))

## Hypothesis 1b — time to reach the reward

Median trial duration over **rewarded trials only** (an unrewarded traverse has no reward time),
and running speed in cm/s. Raw seconds are comparable only *within* a maze stage, because track
length changes from ~48 vu to ~200 vu along the ladder; speed is the cross-stage readout.

**Reported as statistics only — the panels for this hypothesis have been removed.** The numbers
still feed the summary table at the end.

In [ ]:
print("Hypothesis 1b — trends across the full training arc (statistics only):")
for metric, label, pred in (('med_time_to_reward_s', 'median time to reward', -1),
                            ('med_speed_cm_s',       'median running speed',  +1)):
    print("  %s  (predicted to %s)" % (label, 'decrease' if pred < 0 else 'increase'))
    for a in ANIMALS:
        g = ss[ss['animal'] == a]
        rho, p, n = spearman_circshift(g[metric].to_numpy())
        record('1b', label, a, rho, p, n, pred)
        print("     %-5s %s" % (a, fmt_rho(rho, p, n)))

print("")
print("within-stage time-to-reward trends (stages with >=5 sessions):")
for a in ANIMALS:
    for st in STAGE_ORDER:
        h = ss[(ss['animal'] == a) & (ss['stage'] == st)]
        if len(h) >= 5:
            rho, p, n = spearman_circshift(h['med_time_to_reward_s'].to_numpy())
            record('1b', 'time to reward (%s)' % st, a, rho, p, n, -1)
            print("   %-5s %-7s %s" % (a, st, fmt_rho(rho, p, n)))

## Hypothesis 2a — do single-trial trajectories become more similar?

**A warning about the correlation framing.** If an animal learns to run straight down the centre,
`x(y)` collapses toward a constant plus noise. Pearson r measures shared *shape*, and a flat line
has none — so trial-to-trial r can fall toward zero *exactly as behaviour becomes maximally
stereotyped*. Correlation alone can report the opposite of the truth here.

So the plotted metric is **dispersion** (SD of x across trials at each y-bin), which decreases
monotonically with stereotypy. Mean pairwise r and mean |x| are still computed and still appear in
the summary table, but they are no longer plotted; where dispersion and correlation disagree
(they do, for JB2) dispersion is the primary metric.

`splithalf_r` (odd vs even trials) is the **noise ceiling**: without it, cross-session
correlations in 2b are uninterpretable.

In [ ]:
def pairwise_r(X):
    """Mean pairwise Pearson r between trials. Elementwise (no BLAS)."""
    Z = X - X.mean(1, keepdims=True)
    s = Z.std(1, keepdims=True); s = np.where(s == 0, np.nan, s); Z = Z / s
    C = (Z[:, None, :] * Z[None, :, :]).sum(-1) / X.shape[1]
    iu = np.triu_indices(len(X), 1)
    return float(np.nanmean(C[iu]))

def pairwise_rms(X):
    D = X[:, None, :] - X[None, :, :]
    iu = np.triu_indices(len(X), 1)
    return float(np.sqrt((D ** 2).mean(-1))[iu].mean())

traj_rows = []
for key, v in TRAJ_C.items():
    a, d, s = key.split("|")
    X, Y = v['X'], v['Y']
    mt = X.mean(0)
    traj_rows.append(dict(
        animal=a, date=d, session=s, n_traj=len(X),
        disp=float(X.std(0).mean()),
        pairwise_rms=pairwise_rms(X),
        pairwise_r=pairwise_r(X),
        mean_abs_x=float(np.abs(mt).mean()),
        mtraj_ptp=float(np.ptp(mt)),
        splithalf_r=_pearson(X[0::2].mean(0), X[1::2].mean(0)),
        speed_r=pairwise_r(Y),
    ))
tj = pd.DataFrame(traj_rows).sort_values(['animal', 'date', 'session']).reset_index(drop=True)
tj['sess_i'] = tj.groupby('animal').cumcount()
TRAJ_ANIMALS = [a for a in ANIMALS if (tj['animal'] == a).sum() >= 5]
print("animals with >=5 usable 200x20 sessions: %s" % TRAJ_ANIMALS)
print("\nsignal check (maze half-width = 9 vu, 1 vu = %.3f cm):" % CM_PER_VU)
print("  trial-to-trial SD of x : median %.3f vu = %.2f cm = %.1f%% of half-width"
      % (tj['disp'].median(), tj['disp'].median() * CM_PER_VU, 100 * tj['disp'].median() / 9))
print("  mean-trajectory p2p    : median %.3f vu  (a flat mean trajectory would be ~0)" % tj['mtraj_ptp'].median())
print("  split-half reliability : median %.3f, %.0f%% of sessions > 0.9"
      % (tj['splithalf_r'].median(), 100 * (tj['splithalf_r'] > 0.9).mean()))
tj.head()

In [ ]:
n = len(TRAJ_ANIMALS)
fig, axes = plt.subplots(1, n, figsize=(4.1 * n, 3.6), sharey=True)
axes = np.atleast_1d(axes)
for ax, a in zip(axes, TRAJ_ANIMALS):
    g = tj[tj['animal'] == a]
    r1, p1, n1 = spearman_circshift(g['disp'].to_numpy())
    record('2a', 'dispersion', a, r1, p1, n1, -1)
    # still recorded for the summary table, no longer plotted
    record('2a', 'mean pairwise r', a, *spearman_circshift(g['pairwise_r'].to_numpy()), +1)
    record('2a', 'speed-profile r', a, *spearman_circshift(g['speed_r'].to_numpy()), +1)

    ax.plot(g['sess_i'], g['disp'], color=BLUE, marker='o')
    ax.set_title("%s\n%s" % (a, fmt_rho(r1, p1, n1)), loc='left', fontsize=9)
    ax.set_xlabel('session (200x20 only)'); int_xaxis(ax)
axes[0].set_ylabel('SD of x across trials (vu)')
fig.suptitle('Hypothesis 2a - trial-to-trial dispersion of the lateral path',
             x=0.01, ha='left', fontsize=11, color=INK)
fig.tight_layout(rect=(0, 0, 1, 0.88))
plt.show()

# All animals overlaid, NORMALISED per animal. Animals sit at very different absolute dispersion
# (JB4 opens near 4.1 vu, JB1 near 2.4), so on a raw axis the between-animal offset swamps the
# within-animal trend. Each animal is therefore divided by its OWN first-three-session mean, so
# every series starts at 1.0 and the y-axis reads as "fraction of early-training dispersion".
# Caveat: that anchoring forces the series together at session 0 by construction, so the apparent
# agreement at the left edge is not evidence of anything -- only the slopes are informative.
_base = {}
for a in TRAJ_ANIMALS:
    g = tj[tj['animal'] == a].sort_values('sess_i')
    _base[a] = float(g['disp'].iloc[:3].mean())
tjn = tj.copy()
tjn['disp_norm'] = tjn['disp'] / tjn['animal'].map(_base)
print("dispersion baseline (mean of first 3 sessions, vu):")
for a in TRAJ_ANIMALS:
    g = tjn[tjn['animal'] == a].sort_values('sess_i')
    print("   %-5s %.3f vu  ->  final 3 sessions at %.2f x baseline"
          % (a, _base[a], float(g['disp_norm'].iloc[-3:].mean())))

fig, ax = plt.subplots(figsize=(7.6, 4.2))
overlay_with_mean(ax, tjn, 'disp_norm', TRAJ_ANIMALS, x_max=SESS_XMAX)
ax.axhline(1.0, color=MUTED, linestyle='--', linewidth=1.2, zorder=1)
ax.text(0.995, 1.0, ' baseline', transform=ax.get_yaxis_transform(), ha='right', va='bottom',
        fontsize=8, color=MUTED)
ax.set_xlabel('session (200x20 only)')
ax.set_ylabel('dispersion / own first-3-session mean')
int_xaxis(ax); ax.legend(loc='lower left', frameon=False)
fig.suptitle('Hypothesis 2a - dispersion, all animals overlaid (normalised per animal)',
             x=0.01, ha='left', fontsize=11, color=INK)
fig.tight_layout(rect=(0, 0, 1, 0.93))
save_fig(fig, 'h2a_dispersion_overlay_normalised')
plt.show()

### Two example sessions from one animal

The session-level dispersion number, unpacked. One **earlier, high-SD** session and one **later,
low-SD** session from the same animal, chosen automatically as the largest within-animal gap
between the worst session in the first half of training and the best in the second half.

In [ ]:
# Pick the animal with the largest early->late drop in dispersion, and one session from each end.
def _traj_key(r):
    return "%s|%s|%s" % (r['animal'], r['date'], r['session'])

_best = None
for a in TRAJ_ANIMALS:
    g = tj[tj['animal'] == a].sort_values('sess_i').reset_index(drop=True)
    if len(g) < 6:
        continue
    h = len(g) // 2
    e = g.loc[g.iloc[:h]['disp'].idxmax()]      # worst (highest SD) session, first half
    l = g.loc[g.iloc[h:]['disp'].idxmin()]      # best (lowest SD) session, second half
    gap = float(e['disp'] - l['disp'])
    if _best is None or gap > _best[0]:
        _best = (gap, a, e, l)

gap, EX_ANIMAL, EX_EARLY, EX_LATE = _best
EX = [('earlier session', EX_EARLY), ('later session', EX_LATE)]
print("example animal: %s   (dispersion gap %.3f vu)" % (EX_ANIMAL, gap))
for lab, r in EX:
    print("   %-16s %s %s  session index %d, %d trials, SD of x = %.3f vu"
          % (lab, r['date'], r['session'], int(r['sess_i']), int(r['n_traj']), r['disp']))

YB = np.linspace(0, 1, N_BIN)
_xall = np.concatenate([TRAJ_C[_traj_key(r)]['X'].ravel() for _, r in EX])
_lim = max(abs(float(_xall.min())), abs(float(_xall.max()))) * 1.05
XLIM = (-_lim, _lim)

TRIAL_COL = '#9ec5f4'
MEAN_COL  = '#104281'

# ONE alpha for every trial line in this figure. It used to be computed per panel from that
# session's trial count, which gave the two panels different stroke-opacity (0.129 vs 0.102) --
# so the lines genuinely had different appearances and Illustrator was right to refuse to select
# them together. Alpha is kept (not flattened) because these lines overlap and the build-up of
# overlapping strokes is what shows trial density.
_ntr_max = max(len(TRAJ_C[_traj_key(r)]['X']) for _, r in EX)
TRIAL_ALPHA = float(np.clip(20.0 / _ntr_max, 0.06, 0.45))
print("trial-line alpha (shared by both panels): %.4f" % TRIAL_ALPHA)

fig, axes = plt.subplots(1, 2, figsize=(7.6, 4.6), sharex=True, sharey=True)
for ax, (lab, r) in zip(axes, EX):
    X = TRAJ_C[_traj_key(r)]['X']
    alpha = TRIAL_ALPHA
    for row in X:
        ax.plot(row, YB, color=TRIAL_COL, linewidth=0.5, alpha=alpha, zorder=1)
    ax.plot(X.mean(0), YB, color=MEAN_COL, linewidth=2.8, zorder=3)
    ax.axvline(0, color=BASELINE, lw=1, zorder=0)
    ax.set_title("%s\n%s %s   SD = %.2f vu, %d trials"
                 % (lab, r['date'], r['session'], r['disp'], len(X)), loc='left', fontsize=9)
    ax.set_xlabel('lateral position x (vu)'); ax.set_xlim(*XLIM)
    ax.grid(False)
axes[0].set_ylabel('position along track (fraction)')
fig.suptitle('Hypothesis 2a - %s: single-trial paths (thin) and the session mean (thick)'
             % EX_ANIMAL, x=0.01, ha='left', fontsize=11, color=INK)
fig.tight_layout(rect=(0, 0, 1, 0.92))
save_fig(fig, 'h2a_JB2_single_trial_paths')
plt.show()

In [ ]:
# The same two sessions, mean +/- 1 SD across trials instead of the individual trial lines.
fig, axes = plt.subplots(1, 2, figsize=(7.6, 4.6), sharex=True, sharey=True)
for ax, (lab, r) in zip(axes, EX):
    X = TRAJ_C[_traj_key(r)]['X']
    m, sd = X.mean(0), X.std(0)
    ax.fill_betweenx(YB, m - sd, m + sd, color=TRIAL_COL, alpha=0.45, lw=0, zorder=1)
    ax.plot(m, YB, color=MEAN_COL, linewidth=2.8, zorder=3)
    ax.axvline(0, color=BASELINE, lw=1, zorder=0)
    ax.set_title("%s\n%s %s   SD = %.2f vu, %d trials"
                 % (lab, r['date'], r['session'], r['disp'], len(X)), loc='left', fontsize=9)
    ax.set_xlabel('lateral position x (vu)'); ax.set_xlim(*XLIM)
    ax.grid(False)
axes[0].set_ylabel('position along track (fraction)')
fig.suptitle('Hypothesis 2a - %s: session mean +/- 1 SD across trials' % EX_ANIMAL,
             x=0.01, ha='left', fontsize=11, color=INK)
fig.tight_layout(rect=(0, 0, 1, 0.92))
plt.show()

## Hypothesis 2b — do session-average trajectories converge?

Correlating every session against *the literal last session* has two artefacts: the final point is
trivially r = 1, and neighbouring sessions share noise. We instead score each session against a
**leave-one-out late template** — the mean of the last three sessions, excluding the session being
scored — and show the full session x session correlation matrix alongside, which is the honest
exploratory view.

The correlation matrix is **clipped to [0, 1]**, which is where essentially every session pair
sits; because the displayed range no longer spans zero it gets a sequential single-hue map rather
than a diverging one. The panel saturates near the top of that range, and that saturation is
itself the finding. The session x session RMSE panel has been removed.

In [ ]:
MEANS = {}
for a in TRAJ_ANIMALS:
    g = tj[tj['animal'] == a]
    keys = ["%s|%s|%s" % (r.animal, r.date, r.session) for r in g.itertuples()]
    MEANS[a] = np.vstack([TRAJ_C[k]['X'].mean(0) for k in keys])

# imshow keeps a square aspect, so the figure needs the extra height for the axis labels
fig, axes = plt.subplots(1, len(TRAJ_ANIMALS), figsize=(3.9 * len(TRAJ_ANIMALS), 4.4))
axes = np.atleast_1d(axes)
for ax, a in zip(axes, TRAJ_ANIMALS):
    M = MEANS[a]; n = len(M)
    C = np.array([[_pearson(M[i], M[k]) for k in range(n)] for i in range(n)])
    iu = np.triu_indices(n, 1)
    print("%-5s session x session r: min %+.3f, %.1f%% of pairs below 0 (clipped in the colourmap)"
          % (a, C.min(), 100 * (C[iu] < 0).mean()))
    im = ax.imshow(C, cmap=SEQ_CMAP, vmin=0, vmax=1, origin='lower')
    ax.set_title('%s — session x session r' % a, loc='left')
    plt.colorbar(im, ax=ax, fraction=0.046)
    ax.set_xlabel('session'); ax.set_ylabel('session'); ax.grid(False)
fig.suptitle('Hypothesis 2b — correlation between session-average trajectories (colour clipped to [0, 1])',
             x=0.01, ha='left', fontsize=11, color=INK)
fig.tight_layout(rect=(0, 0.02, 1, 0.92))
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(TRAJ_ANIMALS), figsize=(4.2 * len(TRAJ_ANIMALS), 3.4))
axes = np.atleast_1d(axes)
for ax, a in zip(axes, TRAJ_ANIMALS):
    M = MEANS[a]; n = len(M); late = list(range(n - 3, n))
    r_loo, d_loo = [], []
    for i in range(n):
        ref = [k for k in late if k != i]
        T = M[ref].mean(0)
        r_loo.append(_pearson(M[i], T))
        d_loo.append(float(np.sqrt(((M[i] - T) ** 2).mean())))
    rho, p, nn = spearman_circshift(np.array(r_loo))
    record('2b', 'r to late template', a, rho, p, nn, +1)
    record('2b', 'RMSE to late template', a, *spearman_circshift(np.array(d_loo)), -1)
    ceil = tj[tj['animal'] == a]['splithalf_r'].median()
    ax.axhline(ceil, color=MUTED, ls='--', lw=1.2)
    ax.text(0.02, ceil, ' noise ceiling (split-half %.2f)' % ceil, transform=ax.get_yaxis_transform(),
            va='bottom', fontsize=7, color=MUTED)
    ax.plot(range(n), r_loo, color=BLUE, marker='o')
    ax.set_title("%s   %s" % (a, fmt_rho(rho, p, nn)), loc='left')
    ax.set_xlabel('session (200x20 only)'); ax.set_ylabel('r to late template'); ax.set_ylim(-1.05, 1.05)
fig.suptitle('Hypothesis 2b — similarity to a leave-one-out late template',
             x=0.01, ha='left', fontsize=11, color=INK)
fig.tight_layout(rect=(0, 0, 1, 0.9))
plt.show()

### Mean trajectory, first vs last sessions

The underlying picture behind the summary numbers: the session-average path down the track, early
vs late, with the trial-to-trial spread as a band.

In [ ]:
fig, axes = plt.subplots(1, len(TRAJ_ANIMALS), figsize=(3.6 * len(TRAJ_ANIMALS), 4.2), sharey=True)
axes = np.atleast_1d(axes)
for ax, a in zip(axes, TRAJ_ANIMALS):
    g = tj[tj['animal'] == a]
    keys = ["%s|%s|%s" % (r.animal, r.date, r.session) for r in g.itertuples()]
    yb = np.linspace(0, 1, N_BIN)
    for lab, kk, col in (('first 3', keys[:3], '#9ec5f4'), ('last 3', keys[-3:], '#104281')):
        X = np.vstack([TRAJ_C[k]['X'] for k in kk])
        m, sd = X.mean(0), X.std(0)
        ax.fill_betweenx(yb, m - sd, m + sd, color=col, alpha=0.22, lw=0)
        ax.plot(m, yb, color=col, label=lab)
    ax.axvline(0, color=BASELINE, lw=1)
    ax.set_title(a, loc='left'); ax.set_xlabel('lateral position x (vu)')
axes[0].set_ylabel('position along track (fraction)')
fig.legend(*axes[0].get_legend_handles_labels(), loc='upper right',
           bbox_to_anchor=(0.995, 0.965), ncol=2)
fig.suptitle('Mean trajectory +/- 1 SD, first vs last three 200x20 sessions',
             x=0.01, ha='left', fontsize=11, color=INK)
fig.tight_layout(rect=(0, 0, 1, 0.93))
plt.show()

## Summary

In [ ]:
res = pd.DataFrame(RESULTS)
res['dir'] = np.where(res['predicted'] > 0, 'increase', 'decrease')
main = res[~res['metric'].str.contains(r'\(', regex=True)]
piv = (main.pivot_table(index=['hypothesis', 'metric', 'dir'], columns='animal',
                        values='rho', aggfunc='first')
           .reindex(columns=ANIMALS))
print("Spearman rho vs session index (prediction in the 'dir' column)\n")
print(piv.round(3).to_string())
print("\nagreement with prediction, by metric:")
for (h, m, d), grp in main.groupby(['hypothesis', 'metric', 'dir']):
    ok = int(grp['supports'].sum()); tot = len(grp)
    print("   %-3s %-24s expect %-8s %d/%d animals" % (h, m, d, ok, tot))
res.to_csv(os.path.join(CACHE_DIR, 'prelim_results.csv'), index=False)
print("\nwrote prelim_results.csv")

## Limitations — read before quoting any of these numbers

* **No group-level significance is possible.** Hypothesis 2 rests on 3 animals; a sign test on
  n=3 bottoms out at p=0.125 even with perfect agreement. The per-animal rho across ~22 sessions
  is the inferential unit. This is an exploratory analysis, not a confirmatory one.
* **The full-arc accuracy trend is confounded by design.** Maze difficulty escalates *because*
  the animal is improving, so the arc mixes learning with task difficulty. The within-stage
  statistics printed under hypothesis 1a are the clean comparison — and the terminal 200x20 stage
  is at ceiling, so the honest reading is that accuracy learning happens in the shaping ladder.
* **`50x5` is excluded from hypothesis 1a by construction, not by judgement.** The reward tower
  spans the full maze there, so every completed traverse is rewarded and accuracy is 100%.
  Including it would have inflated the start of the arc with a number that cannot vary.
* **Dispersion and correlation can disagree**, and here they do (JB2). Dispersion is the primary
  metric because correlation is vulnerable to the flat-line artefact; where they conflict, check
  `mean_abs_x` in the `tj` table and the mean-trajectory panels before concluding anything.
* **`accuracy_last50` carries a censoring bias.** Sessions end either at the ~200-reward cap or at
  manual Esc, so the last 50 trials of a capped session are when the animal was performing well,
  while the last 50 of an uncapped session are when the experimenter gave up. Never use it as the
  primary accuracy measure.
* **Session index conflates calendar time with training amount** (JB1 has a 7-day gap around
  260623-260707). The robustness check below re-runs the headline trends against cumulative trials.
* **Trial count is an endogenous denominator** — a disengaged animal accrues trials without
  traversing productively, which deflates accuracy for reasons unrelated to strategy.
* **Hypothesis 2 is 200x20-only.** Maze width changes along the ladder (5/10/20 vu), and in a
  narrow maze the animal physically cannot deviate, so early stages would look artificially
  stereotyped. Normalising x by half-width does not remove this, so cross-stage dispersion is not
  reported at all.

In [ ]:
# NOTE: Spearman is rank-based and cumulative trials is monotonic in session index, so the two
# give IDENTICAL rho by construction -- a rank correlation cannot test this at all. Unequal
# spacing only shows up in a spacing-sensitive statistic, so this uses Pearson on raw values.
print("Robustness: session index vs CUMULATIVE TRIALS as x-axis (Pearson, spacing-sensitive)")
for a in ANIMALS:
    g = ss[ss['animal'] == a]
    print("   %-5s accuracy r=%+.3f (vs sess %+.3f)   speed r=%+.3f (vs sess %+.3f)" % (
        a,
        _pearson(g['cum_trials'], g['accuracy']), _pearson(g['sess_i'], g['accuracy']),
        _pearson(g['cum_trials'], g['med_speed_cm_s']), _pearson(g['sess_i'], g['med_speed_cm_s'])))
print("")
print("(Spearman is unchanged by this swap by construction -- that is why Pearson is used here.)")

# Hypothesis 3 — errors in executing the stereotyped trajectory

Two independent notions of "error", deliberately kept separate because they can dissociate:

1. **Deviation** — the trial's path departs from the animal's usual route.
2. **Anti-goal motion** — the animal moves *away* from the reward tower.

**Why heading is not used.** `tmaze.py:401 compute_goal_variables` already implements a full
goal/heading instrument (`h_target`, `h_error`, `abs_h_error`, `dh_error`). Its *distance* half
ports here; its *heading* half does not, because **`h_int` and `dh` are identically zero in every
`wideLinearTrack` session** — the view angle is locked, the animal translates but cannot turn.
With `h == 0`, `h_error` reduces to `-h_target`, a deterministic function of position that
carries no behavioural information. Goal-directedness is therefore computed from the **velocity
vector**, which is what `approach_rate` in the trial table does.

---

### ⚠ FLAGGED, NOT FIXED — the `dx` sign, and what it does to `approach_rate`

`load_vr` flips lateral position (`df['x'] = -df['x']`, following `wideLinearTrack.py:54`) but
leaves `dx` untouched. Measured on raw `sessionData`, `diff(x_raw) / (dx*dt) = +1.000` exactly
(checked on JB1 260612 and JB4 260716), so **`dx` tracks the *unflipped* `x`** — as loaded, the
two disagree about which direction is rightward.

That matters for this section, because goal-directedness is built from the velocity vector:

```
tdist = hypot(xx, y_goal - yy)
appr  = ((y_goal - yy) * uy - xx * ux) / tdist        # ux = dx, uy = dy
```

The `- xx * ux` term is the lateral contribution to closing on the tower. With `xx` flipped and
`ux` not, that term enters with the **wrong sign**, so `approach_rate` is wrong, and with it
`frac_time_toward`, `frac_time_backward`, and the H3b results in `prelim_results_h3.csv`.

**How much it matters, honestly:** not uniformly. The forward term `(y_goal - yy) * uy` is order
`200 × 40` while the lateral term is order `10 × 3`, so over most of the track the error is a
small perturbation on a large positive number and the *sign* of `appr` is unaffected. It bites
where the two terms are comparable — **near the tower**, where `y_goal - yy → 0` and `tdist` is
small. Since `frac_time_toward` thresholds on `sign(appr)`, the trials it can flip are exactly
the ones where the animal is close to the goal and barely approaching.

**Deliberately left unchanged** so the numbers in this section still reproduce
`prelim_results_h3.csv` as published. H4 corrects the sign in its own per-sample cache and pins
it with a QC assertion (`median diff(x)/(dx*dt)` must be `+1`). Fix here before leaning on H3b.


In [ ]:
print("QC — view angle must be locked (h_int identically 0) for the heading argument to hold:")
print("   max session std(h_int) = %.6f" % trials['h_std'].max())
assert trials['h_std'].max() == 0, "view angle is NOT locked; heading metrics become available"

print("\nQC — dx/dy are velocities, so approach_rate is valid (diff(y) == dy*dt):")
_a, _d, _s = SESSIONS[0]
_df, _ = load_vr(_a, _d, _s)
_y = _df['y'].to_numpy(); _vy = _df['dy'].to_numpy(); _dt = _df['dt'].to_numpy()
_de = np.diff(_y); _dp = (_vy[:-1] * _dt[:-1]); _ok = np.abs(_dp) > 1e-6
_ratio = float(np.median(_de[_ok] / _dp[_ok]))
print("   %s/%s median diff(y)/(dy*dt) = %.4f" % (_a, _d, _ratio))
assert abs(_ratio - 1.0) < 0.05, "dx/dy are not velocities; approach_rate formula is wrong"

print("\nper-trial goal-directedness, pooled by maze stage:")
print(kept.groupby('stage')[['frac_time_toward', 'frac_time_backward', 'n_reversals',
                             'straightness']].median().reindex(STAGE_ORDER).round(3).to_string())

## Scoring every trial against a template

Two references, per the design:

1. **Late/expert template** *(primary)* — mean of the animal's last 3 sessions on 200x20,
   **leave-one-out** so a session is never scored against a template containing itself.
2. **Session's own leave-one-out mean** — "what was unusual that day". Non-circular.

**The error threshold is fixed across sessions, and this is not a detail.** If an error were
defined as "> 2 SD from this session's mean", roughly 5% of trials would be errors in *every*
session by construction and hypothesis 3a would be flat no matter what the animal did. The
threshold is therefore the **90th percentile of `dev_rms` pooled across all of that animal's
200x20 trials**, computed once. The animal's overall error rate is 10% by construction; the
per-session rate is free to move, which is what makes the trend meaningful.

In [ ]:
ERR_PCTL = 90

dev_rows = []
for a in TRAJ_ANIMALS:
    keys = sorted([k for k in TRAJ if k.startswith(a + "|")], key=lambda k: k.split("|")[1:])
    means = {k: TRAJ[k]['X'].mean(0) for k in keys}
    late_keys = keys[-3:]
    for si, k in enumerate(keys):
        v = TRAJ[k]; X = v['X']
        # reference 1: leave-one-out late template
        ref_late = np.mean([means[j] for j in late_keys if j != k], axis=0)
        # reference 2: this session's own leave-one-out mean (per trial)
        n = len(X); tot = X.sum(0)
        for i in range(n):
            own = (tot - X[i]) / max(n - 1, 1)
            d_late = X[i] - ref_late
            d_own = X[i] - own
            dev_rows.append(dict(
                animal=a, key=k, date=k.split("|")[1], session=k.split("|")[2],
                sess_i=si, trial=int(v['tr'][i]),
                rewarded=bool(v['rew'][i]), max_backtrack=float(v['bt'][i]),
                dev_rms=float(np.sqrt((d_late ** 2).mean())),
                dev_max=float(np.abs(d_late).max()),
                dev_argmax_bin=int(np.abs(d_late).argmax()),
                dev_rms_own=float(np.sqrt((d_own ** 2).mean())),
            ))
dev = pd.DataFrame(dev_rows)

THRESH = {}
for a in TRAJ_ANIMALS:
    g = dev[dev['animal'] == a]
    THRESH[a] = dict(late=float(np.percentile(g['dev_rms'], ERR_PCTL)),
                     own=float(np.percentile(g['dev_rms_own'], ERR_PCTL)))
dev['is_error'] = [r.dev_rms > THRESH[r.animal]['late'] for r in dev.itertuples()]
dev['is_error_own'] = [r.dev_rms_own > THRESH[r.animal]['own'] for r in dev.itertuples()]

print("per-animal fixed thresholds (vu) at the %dth percentile, pooled over all sessions:" % ERR_PCTL)
for a in TRAJ_ANIMALS:
    print("   %-5s late-template %.3f vu (%.2f cm)   own-mean %.3f vu" % (
        a, THRESH[a]['late'], THRESH[a]['late'] * CM_PER_VU, THRESH[a]['own']))
print("\ntrials scored: %d   overall error rate %.3f (10%% by construction)"
      % (len(dev), dev['is_error'].mean()))

## Validation — does deviation actually predict failure?

The task supplies a ground-truth error label for free: an **unrewarded trial** means the animal
was not centred enough at the tower to trigger reward. If the deviation metric is measuring
task-relevant error, high-deviation trials should be disproportionately unrewarded.

**This check can fail, and if it does the metric is wrong, not the check.** An AUC near 0.5 would
mean `dev_rms` is picking up harmless route variation rather than error, and hypothesis 3a would
have to be reported as "variability", not "errors".

In [ ]:
def auc(score, label):
    """Mann-Whitney AUC: P(score of a positive > score of a negative). No BLAS, no sklearn."""
    score = np.asarray(score, float); label = np.asarray(label, bool)
    n1, n0 = int(label.sum()), int((~label).sum())
    if n1 == 0 or n0 == 0:
        return np.nan
    r = _rank(score) + 1.0
    return float((r[label].sum() - n1 * (n1 + 1) / 2.0) / (n1 * n0))

fig, axes = plt.subplots(1, len(TRAJ_ANIMALS), figsize=(4.0 * len(TRAJ_ANIMALS), 3.4), sharey=True)
axes = np.atleast_1d(axes)
for ax, a in zip(axes, TRAJ_ANIMALS):
    g = dev[dev['animal'] == a].copy()
    fail = ~g['rewarded'].to_numpy()
    A = auc(g['dev_rms'].to_numpy(), fail)
    q = pd.qcut(g['dev_rms'], 10, labels=False, duplicates='drop')
    p_fail = g.assign(q=q).groupby('q')['rewarded'].apply(lambda s: 1 - s.mean())
    ax.plot(p_fail.index + 1, 100 * p_fail.values, color=CRITICAL, marker='o')
    ax.axhline(100 * fail.mean(), color=MUTED, ls='--', lw=1.2)
    ax.text(0.02, 100 * fail.mean(), ' base rate', transform=ax.get_yaxis_transform(),
            va='bottom', fontsize=7, color=MUTED)
    ax.set_title("%s   AUC = %.3f" % (a, A), loc='left')
    ax.set_xlabel('deviation decile (1 = closest to template)')
    int_xaxis(ax)
    record('3-val', 'AUC dev predicts failure', a, A, np.nan, len(g), +1)
axes[0].set_ylabel('% of trials unrewarded')
fig.suptitle('Validation — do high-deviation trials fail more often?',
             x=0.01, ha='left', fontsize=11, color=INK)
fig.tight_layout(rect=(0, 0, 1, 0.9))
plt.show()

print("AUC by animal (0.5 = deviation carries no information about reward failure):")
for a in TRAJ_ANIMALS:
    g = dev[dev['animal'] == a]
    print("   %-5s AUC=%.3f   n=%d trials, %.1f%% unrewarded"
          % (a, auc(g['dev_rms'], ~g['rewarded']), len(g), 100 * (~g['rewarded']).mean()))

## Example error trials

The highest-deviation trials, plotted against the template they depart from. The path is coloured
by the sign of `approach_rate`, so stretches where the animal was moving **away** from the reward
tower are directly visible. Per-sample data for these few trials is re-loaded from `Y:` on demand
rather than caching every raw sample.

In [ ]:
# Per-sample data for the example-trial figures. The trajectory cache keeps only 40 binned points
# per trial, so these figures need the raw samples back -- and re-reading session .mat files from
# the slow share is what dominates this notebook's runtime. Cached in memory AND on disk, so a
# rerun (e.g. after a plotting tweak) costs nothing.
SESS_PKL = os.path.join(CACHE_DIR, 'prelim_cache_sessions.pkl')

def _load_sess_cache():
    if os.path.exists(SESS_PKL):
        try:
            with open(SESS_PKL, 'rb') as f:
                return pickle.load(f)
        except Exception as e:
            print('session cache unreadable (%s) - rebuilding' % e)
    return {}

_SESS_CACHE = _load_sess_cache()

def session_all_trials(animal, date, sess):
    """{trial_id: (x, y, dx, dy)} for every trial of one session, plus y-bin centres and y_goal."""
    key = "%s|%s|%s" % (animal, date, sess)
    if key in _SESS_CACHE:
        return _SESS_CACHE[key]
    df, _meta = load_vr(animal, date, sess)
    x  = df['x'].to_numpy().astype(np.float32);  y  = df['y'].to_numpy().astype(np.float32)
    vx = df['dx'].to_numpy().astype(np.float32); vy = df['dy'].to_numpy().astype(np.float32)
    tr = df['trial'].to_numpy()
    edges = np.flatnonzero(np.diff(tr) != 0) + 1
    st = np.concatenate([[0], edges]); sp = np.concatenate([edges, [len(tr)]])
    trials_xy = {int(tr[a]): (x[a:b], y[a:b], vx[a:b], vy[a:b])
                 for a, b in zip(st, sp) if b - a >= 3}
    y_goal = float(y.max())
    y_start_med = float(np.median([y[a] for a, b in zip(st, sp) if b - a >= 3]))
    be = np.linspace(y_start_med + 2.0, y_goal - 2.0, N_BIN + 1)
    out = (trials_xy, 0.5 * (be[:-1] + be[1:]), y_goal)
    _SESS_CACHE[key] = out
    with open(SESS_PKL, 'wb') as f:
        pickle.dump(_SESS_CACHE, f, protocol=4)
    return out

def trial_samples(animal, date, sess, trial_id):
    """Per-sample x, y and approach rate for one trial, plus the y-bin centres in vu
    so the template can be drawn on the SAME axis as the trial."""
    trials_xy, centres, y_goal = session_all_trials(animal, date, sess)
    xx, yy, ux, uy = trials_xy[int(trial_id)]
    tdist = np.hypot(xx, y_goal - yy)
    appr = ((y_goal - yy) * uy - xx * ux) / np.maximum(tdist, 1e-6)
    return xx, yy, appr, y_goal, centres

N_EX = 3
for a in TRAJ_ANIMALS:
    g = dev[dev['animal'] == a]
    keys = sorted(set(g['key']), key=lambda k: k.split("|")[1:])
    early_k, late_k = keys[:len(keys) // 2], keys[len(keys) // 2:]
    picks_hi = []
    for lbl, ks in (('early', early_k), ('late', late_k)):
        sub = g[g['key'].isin(ks)].nlargest(N_EX, 'dev_rms')
        picks_hi += [(lbl, r) for r in sub.itertuples()]
    ref = {k: TRAJ[k]['X'].mean(0) for k in keys}
    late_ref = np.mean([ref[k] for k in keys[-3:]], axis=0)

    fig, axes = plt.subplots(1, len(picks_hi), figsize=(2.5 * len(picks_hi), 4.6), sharey=True)
    axes = np.atleast_1d(axes)
    for ax, (lbl, r) in zip(axes, picks_hi):
        an, dd_, ss_ = r.key.split("|")
        xx, yy, appr, y_goal, centres = trial_samples(an, dd_, ss_, r.trial)
        ax.plot(late_ref, centres, color=MUTED, lw=2, label='usual route')
        tw = appr > 0
        ax.scatter(xx[tw], yy[tw], s=3, color=BLUE, label='toward tower')
        ax.scatter(xx[~tw], yy[~tw], s=3, color=CRITICAL, label='away from tower')
        ax.set_title("%s  d=%.1f vu\n%s tr%d%s" % (lbl, r.dev_rms, dd_, r.trial,
                     "" if r.rewarded else "  NO REWARD"), loc='left', fontsize=8)
        ax.set_xlim(-10, 10); ax.set_xlabel('x (vu)')
    axes[0].set_ylabel('position along track y (vu)  -  tower at top')
    fig.legend(*axes[0].get_legend_handles_labels(), loc='upper right',
               bbox_to_anchor=(0.995, 0.97), ncol=3, fontsize=8)
    fig.suptitle('%s - highest-deviation trials' % a, x=0.01, ha='left', fontsize=11, color=INK)
    fig.tight_layout(rect=(0, 0, 1, 0.90))
    plt.show()

## Transient errors: leave the route, then correct back onto it

The "highest deviation" trials above are ranked by RMS deviation over the whole traverse, so they
are dominated by trials that are wrong for their entire length -- a wall-hug from start to finish.
That is a *sustained* departure, not an error-and-correction.

This cell looks for the other pattern: a trial that is **following** the usual route, **departs**
from it over a bounded stretch of track, and then **rejoins** it and carries on. Selection:

* exactly **one** contiguous stretch where `|deviation| > EXC_THR`
* that stretch touches **neither end** of the track, so there is both an approach and a recovery
* **outside** it, mean `|deviation| < BASE_MAX` -- i.e. the rest of the trial really is on-route

Thresholds come from the measured deviation distribution (typical trial-to-trial dispersion is
~2.7 vu), not from guesswork. Note this is a *shape* criterion, not an outcome one: a trial that
corrects successfully often still earns its reward, so these are not the same population as the
unrewarded trials.

In [ ]:
EXC_THR   = 2.5        # vu -- deviation above this counts as off-route
BASE_MAX  = 1.0        # vu -- mean |deviation| outside the excursion must be tighter than this
EDGE_BINS = 3          # excursion may not start/end within this many bins of either track end
EXC_WIDTH = (2, 20)    # bins; wider than this is a sustained departure, not a transient one

def _runs(mask):
    """Contiguous True runs as (start, end) inclusive bin indices."""
    out = []; s = None
    for i, v in enumerate(mask):
        if v and s is None:
            s = i
        if not v and s is not None:
            out.append((s, i - 1)); s = None
    if s is not None:
        out.append((s, len(mask) - 1))
    return out

exc_rows = []
for a in TRAJ_ANIMALS:
    keys = sorted([k for k in TRAJ if k.startswith(a + "|")], key=lambda k: k.split("|")[1:])
    means = {k: TRAJ[k]['X'].mean(0) for k in keys}
    late = keys[-3:]
    for si, k in enumerate(keys):
        ref = np.mean([means[j] for j in late if j != k], axis=0)
        v = TRAJ[k]
        for i in range(len(v['X'])):
            d = np.abs(v['X'][i] - ref)
            rr = [r for r in _runs(d > EXC_THR)
                  if r[0] >= EDGE_BINS and r[1] <= N_BIN - 1 - EDGE_BINS]
            if len(rr) != 1:
                continue
            s, e = rr[0]
            w = e - s + 1
            if not (EXC_WIDTH[0] <= w <= EXC_WIDTH[1]):
                continue
            outside = np.concatenate([d[:max(s - 1, 0)], d[min(e + 2, N_BIN):]])
            if len(outside) < 10:
                continue
            base = float(outside.mean())
            if base > BASE_MAX:
                continue
            exc_rows.append(dict(
                animal=a, key=k, date=k.split("|")[1], session=k.split("|")[2], sess_i=si,
                trial=int(v['tr'][i]), rewarded=bool(v['rew'][i]),
                bin_start=s, bin_end=e, width=w,
                peak=float(d[s:e + 1].max()), peak_bin=int(s + np.argmax(d[s:e + 1])),
                base=base, prominence=float(d[s:e + 1].max() - base),
                signed_peak=float((v['X'][i] - ref)[s + int(np.argmax(d[s:e + 1]))]),
            ))
exc = pd.DataFrame(exc_rows)
print("transient error-and-correction trials: %d of %d scored (%.1f%%)"
      % (len(exc), len(dev), 100.0 * len(exc) / len(dev)))
print(exc.groupby('animal').agg(n=('trial', 'size'), median_peak=('peak', 'median'),
                                median_width_bins=('width', 'median'),
                                frac_rewarded=('rewarded', 'mean')).round(3).to_string())
print("\nexcursion width in virmen units: median %.1f vu of a ~196 vu track"
      % (exc['width'].median() * (196.0 / N_BIN)))

# --- plot the clearest examples -----------------------------------------------------------
# Ranked by prominence (peak deviation minus the on-route baseline), so the excursion stands out
# against an otherwise template-following trial. Per-sample data is reloaded for these few trials.
N_EX_PER_ANIMAL = 2
picks = []
for a in TRAJ_ANIMALS:
    picks += list(exc[exc['animal'] == a].nlargest(N_EX_PER_ANIMAL, 'prominence').itertuples())

fig, axes = plt.subplots(1, len(picks), figsize=(2.3 * len(picks), 5.4), sharey=True)
axes = np.atleast_1d(axes)
for ax, r in zip(axes, picks):
    an, dd_, ss_ = r.key.split("|")
    xx, yy, _appr, y_goal, centres = trial_samples(an, dd_, ss_, r.trial)
    keys = sorted([k for k in TRAJ if k.startswith(an + "|")], key=lambda k: k.split("|")[1:])
    ref = np.mean([TRAJ[j]['X'].mean(0) for j in keys[-3:] if j != r.key], axis=0)

    y_lo, y_hi = centres[r.bin_start], centres[r.bin_end]
    ax.axhspan(y_lo, y_hi, color='#f2b3b2', alpha=0.35, lw=0, zorder=0)
    ax.plot(ref, centres, color=MUTED, lw=1.8, zorder=2, label='usual route')
    ax.plot(xx, yy, color=BLUE, lw=1.3, zorder=3, label='example trial')
    ax.axvline(0, color=BASELINE, lw=0.8, zorder=1)
    ax.set_title("%s %s tr%d\npeak %.1f vu %s\nbase %.1f vu%s"
                 % (an, dd_, r.trial, r.peak, 'right' if r.signed_peak > 0 else 'left',
                    r.base, "" if r.rewarded else "  NO REWARD"),
                 loc='left', fontsize=7.5)
    ax.set_xlim(-10, 10); ax.set_ylim(0, y_goal)
    ax.set_xlabel('x (vu)'); ax.grid(False)
axes[0].set_ylabel('position along track y (vu)  -  tower at top')
fig.legend(*axes[0].get_legend_handles_labels(), loc='upper right',
           bbox_to_anchor=(0.995, 0.975), ncol=2, fontsize=8)
fig.suptitle('Transient errors: shaded band marks the off-route stretch, before and after which\n'
             'the trial sits on its usual route', x=0.01, ha='left', fontsize=10, color=INK)
fig.tight_layout(rect=(0, 0, 1, 0.90))
save_fig(fig, 'h3_transient_error_corrections')
plt.show()

### The same error trials against normal trials from the same session

Each panel takes one error trial from the figure above and draws it against **10 ordinary trials
from its own session**, so the excursion can be judged against what that animal was doing on that
day rather than against a template averaged over three late sessions.

The 10 comparison trials are **rewarded trials from the same session that were not themselves
flagged as excursions**, sampled at even intervals across the session rather than picked for
tidiness -- so they show the genuine spread of the animal's normal route, not a best-of selection.

In [ ]:
from matplotlib.lines import Line2D

N_COMPARE = 10
NORMAL_COL = '#9ec5f4'     # ordinary rewarded trials
UNREW_COL  = '#8f1f1e'     # unrewarded trial: deeper red than the error trial

fig, axes = plt.subplots(1, len(picks), figsize=(2.3 * len(picks), 5.4), sharey=True)
axes = np.atleast_1d(axes)
print("%-22s %8s %8s  %s" % ("panel", "normal", "unrew", "unrewarded trial shown"))
for ax, r in zip(axes, picks):
    an, dd_, ss_ = r.key.split("|")
    trials_xy, centres, y_goal = session_all_trials(an, dd_, ss_)

    # ordinary trials from this session: rewarded, and not themselves flagged as excursions
    v = TRAJ[r.key]
    flagged = set(exc[exc['key'] == r.key]['trial'])
    pool = [int(t) for t, rw in zip(v['tr'], v['rew'])
            if rw and int(t) not in flagged and int(t) in trials_xy]
    if len(pool) > N_COMPARE:
        pool = [pool[i] for i in np.linspace(0, len(pool) - 1, N_COMPARE).astype(int)]

    # one UNREWARDED trial from the same session, chosen at the MEDIAN of their deviation
    # distribution rather than the worst -- an extreme pick would misrepresent what a failed
    # traverse typically looks like on this day.
    dsub = dev[(dev['key'] == r.key) & (~dev['rewarded']) & (dev['trial'] != r.trial)]
    dsub = dsub[dsub['trial'].isin(list(trials_xy))]
    pick_un = None
    if len(dsub):
        gap = (dsub['dev_rms'] - dsub['dev_rms'].median()).abs().to_numpy()
        pick_un = int(dsub.iloc[int(gap.argmin())]['trial'])

    y_lo, y_hi = centres[r.bin_start], centres[r.bin_end]
    ax.axhspan(y_lo, y_hi, color='#f2b3b2', alpha=0.35, lw=0, zorder=0)
    for t in pool:
        xs, ys, _, _ = trials_xy[t]
        ax.plot(xs, ys, color=NORMAL_COL, lw=0.9, zorder=2)

    Xc = TRAJ_C.get(r.key, TRAJ[r.key])['X']
    ax.plot(Xc.mean(0), centres, color=MEAN_LINE, lw=2.4, zorder=3)

    if pick_un is not None:
        xs, ys, _, _ = trials_xy[pick_un]
        ax.plot(xs, ys, color=UNREW_COL, lw=1.4, zorder=4)

    xs, ys, _, _ = trials_xy[r.trial]
    ax.plot(xs, ys, color=CRITICAL, lw=1.8, zorder=5)
    ax.axvline(0, color=BASELINE, lw=0.8, zorder=1)
    ax.set_title("%s %s tr%d\npeak %.1f vu %s%s"
                 % (an, dd_, r.trial, r.peak, 'right' if r.signed_peak > 0 else 'left',
                    "" if r.rewarded else "\nNO REWARD"), loc='left', fontsize=7.5)
    ax.set_xlim(-10, 10); ax.set_ylim(0, y_goal)
    ax.set_xlabel('x (vu)'); ax.grid(False)
    print("%-22s %8d %8d  %s"
          % ("%s %s tr%d" % (an, dd_, r.trial), len(pool), len(dsub),
             ("tr%d" % pick_un) if pick_un is not None else "none in session"))

axes[0].set_ylabel('position along track y (vu)  -  tower at top')
# Proxy handles: a panel may have no unrewarded trial, so the legend must not depend on what
# happens to have been drawn in the first axes.
handles = [Line2D([], [], color=NORMAL_COL, lw=0.9, label='normal rewarded trials'),
           Line2D([], [], color=MEAN_LINE,  lw=2.4, label='session mean'),
           Line2D([], [], color=UNREW_COL,  lw=1.4, label='unrewarded trial'),
           Line2D([], [], color=CRITICAL,   lw=1.8, label='error trial')]
fig.legend(handles=handles, loc='upper right', bbox_to_anchor=(0.995, 0.975),
           ncol=4, fontsize=8)
fig.suptitle("Error trial (red) against ordinary trials, an unrewarded trial, "
             "and the day's mean route", x=0.01, ha='left', fontsize=10, color=INK)
fig.tight_layout(rect=(0, 0, 1, 0.91))
save_fig(fig, 'h3_transient_error_vs_normal_trials')
plt.show()

## Hypothesis 3a — do deviations become less common over training?

Fraction of trials flagged as errors, per session, on 200x20 (JB1/JB2/JB4). Both references shown.

**Caveat on the primary reference.** Late sessions sit closer to a late template almost by
construction, so some downward trend is favoured even with leave-one-out. The session-own-mean
panel is the non-circular cross-check — but note it is close to a discretised version of the
`disp` metric from hypothesis 2a, so it is a cross-check rather than independent evidence.

In [ ]:
err = (dev.groupby(['animal', 'key', 'sess_i'], as_index=False)
          .agg(frac_error=('is_error', 'mean'), frac_error_own=('is_error_own', 'mean'),
               med_dev=('dev_rms', 'median'), n=('trial', 'size'))
          .sort_values(['animal', 'sess_i']))

fig, axes = plt.subplots(2, len(TRAJ_ANIMALS), figsize=(4.0 * len(TRAJ_ANIMALS), 6.0), sharex='col')
axes = np.atleast_2d(axes).reshape(2, len(TRAJ_ANIMALS))
for j, a in enumerate(TRAJ_ANIMALS):
    g = err[err['animal'] == a]
    r1, p1, n1 = spearman_circshift(g['frac_error'].to_numpy())
    r2, p2, n2 = spearman_circshift(g['frac_error_own'].to_numpy())
    record('3a', 'error rate (late template)', a, r1, p1, n1, -1)
    record('3a', 'error rate (own mean)', a, r2, p2, n2, -1)
    axes[0, j].plot(g['sess_i'], 100 * g['frac_error'], color=BLUE, marker='o')
    axes[0, j].set_title("%s\nvs late template  %s" % (a, fmt_rho(r1, p1, n1)), loc='left', fontsize=9)
    axes[0, j].set_ylabel('% trials flagged as errors')
    axes[1, j].plot(g['sess_i'], 100 * g['frac_error_own'], color=ORANGE, marker='o')
    axes[1, j].set_title("vs session's own mean  %s" % fmt_rho(r2, p2, n2), loc='left', fontsize=9)
    axes[1, j].set_ylabel('% trials flagged as errors')
    axes[1, j].set_xlabel('session (200x20 only)')
    for ax in (axes[0, j], axes[1, j]):
        ax.axhline(100 * (1 - ERR_PCTL / 100.0), color=MUTED, ls='--', lw=1)
        int_xaxis(ax)
fig.suptitle('Hypothesis 3a — error rate across training (dashed = pooled rate, fixed by threshold)',
             x=0.01, ha='left', fontsize=11, color=INK)
fig.tight_layout(rect=(0, 0, 1, 0.94))
plt.show()

print("threshold sanity — per-session rates must VARY, else the threshold was re-estimated per session:")
for a in TRAJ_ANIMALS:
    g = err[err['animal'] == a]
    print("   %-5s pooled %.3f   per-session SD %.4f" % (a, g['frac_error'].mean(), g['frac_error'].std()))
    assert g['frac_error'].std() > 0, "per-session error rate is constant -- threshold bug"

## Hypothesis 3b — do animals spend more time moving toward the tower?

Unlike lateral dispersion, these metrics are **dimensionless and not confounded by maze width**,
so they run across the whole shaping ladder for all four animals — including JB9, which the
trajectory analyses had to drop for lack of 200x20 sessions.

> ⚠ `frac_time_toward` and `frac_time_backward` both derive from `approach_rate`, whose lateral
> term carries the wrong sign — see the flag at the end of the Hypothesis 3 introduction. The
> effect is small away from the tower and largest close to it. Not fixed here.


In [ ]:
gd = (kept.groupby(['animal', 'date', 'session'], as_index=False)
          .agg(stage=('stage', 'first'),
               frac_time_toward=('frac_time_toward', 'mean'),
               straightness=('straightness', 'median'),
               n_reversals=('n_reversals', 'median'))
          .sort_values(['animal', 'date', 'session']))
gd['sess_i'] = gd.groupby('animal').cumcount()

SPECS = [('frac_time_toward', 'fraction of time moving toward tower', +1, BLUE),
         ('straightness', 'path straightness (net / path length)', +1, ORANGE),
         ('n_reversals', 'median reversals per trial', -1, CRITICAL)]
fig, axes = plt.subplots(3, len(ANIMALS), figsize=(3.4 * len(ANIMALS), 8.8), sharex='col', sharey='row')
for j, a in enumerate(ANIMALS):
    g = gd[gd['animal'] == a]
    for i, (col, lab, pred, colr) in enumerate(SPECS):
        ax = axes[i, j]
        shade_stages(ax, list(g['stage']))
        ax.plot(g['sess_i'], g[col], color=colr, marker='o', zorder=3, markersize=4)
        rho, p, n = spearman_circshift(g[col].to_numpy())
        record('3b', lab, a, rho, p, n, pred)
        ax.set_title(("%s\n" % a if i == 0 else "") + fmt_rho(rho, p, n), loc='left', fontsize=8)
        if j == 0:
            ax.set_ylabel(lab, fontsize=7)
        int_xaxis(ax)
    axes[-1, j].set_xlabel('session (full arc)')
stage_legend(fig, set(gd['stage'].dropna()))
fig.suptitle('Hypothesis 3b — goal-directed motion across the full training arc',
             x=0.01, ha='left', fontsize=11, color=INK)
fig.tight_layout(rect=(0, 0.07, 1, 0.95))
plt.show()

### Is the 3b trend just the maze getting longer?

`frac_time_toward` is dimensionless, but it is **not** stage-independent — it varies with maze
stage (pooled medians run 0.81-0.93 across the ladder, with 200x20 the lowest). Since animals
move onto 200x20 partway through the arc, a step change at that transition could manufacture the
full-arc trend. This checks the trend **within the terminal maze alone**, where no such step exists.

In [ ]:
print("3b within-stage trends (stages with >=5 sessions)")
print("%-5s %-8s %4s %9s %10s %9s" % ("animal", "stage", "n", "toward", "straight", "revers"))
for a in ANIMALS:
    for st in STAGE_ORDER:
        h = gd[(gd['animal'] == a) & (gd['stage'] == st)]
        if len(h) < 5:
            continue
        i = np.arange(len(h), dtype=float)
        print("%-5s %-8s %4d %9.3f %10.3f %9.3f" % (
            a, st, len(h), spearman(i, h['frac_time_toward']),
            spearman(i, h['straightness']), spearman(i, h['n_reversals'])))
        if st == '200x20' and len(h) >= 10:
            rho, p, n = spearman_circshift(h['frac_time_toward'].to_numpy())
            record('3b', 'frac time toward (200x20 only)', a, rho, p, n, +1)
print()
print("mean frac_time_toward by animal x stage (the step the check is guarding against):")
print(gd.pivot_table(index='animal', columns='stage', values='frac_time_toward')
        .reindex(columns=STAGE_ORDER).round(3).to_string())

## Hypothesis 3 summary

In [ ]:
res3 = pd.DataFrame([r for r in RESULTS if r['hypothesis'].startswith('3')])
piv3 = (res3.pivot_table(index=['hypothesis', 'metric'], columns='animal', values='rho', aggfunc='first')
            .reindex(columns=ANIMALS))
print(piv3.round(3).to_string())
print("\nagreement with prediction:")
for (h, m), grp in res3[res3['hypothesis'] != '3-val'].groupby(['hypothesis', 'metric']):
    print("   %-4s %-38s %d/%d animals" % (h, m, int(grp['supports'].sum()), len(grp)))
res3.to_csv(os.path.join(CACHE_DIR, 'prelim_results_h3.csv'), index=False)
print("\nwrote prelim_results_h3.csv")

## Hypothesis 4 — the timecourse of correction behaviour

Everything in H3 is measured against **track position**: the 40 y-bins of `TRAJ['X']`. That
answers *where* a trial leaves its route, but not *how long* the animal takes to peel away, nor
how long the correction itself takes. Those are questions about seconds, and seconds were thrown
away when the trajectory cache was binned.

H4 rebuilds a per-sample, time-resolved layer and re-detects the H3 excursions on it.

Two constraints from the data shape everything below:

* **View angle is locked** (`h_std == 0`, asserted in the H3 QC cell). There is no heading signal,
  so a "correction" here is purely lateral translation — there is no turn to measure.
* **`dx` is lateral velocity in vu/s**, already recorded (`diff(y) == dy*dt`, asserted in the same
  cell). The rate at which an animal corrects therefore needs no numerical differentiation, which
  matters at ~56 Hz where differentiating position would be mostly noise.

**The order here is deliberate.** Single-trial verification (h4a, h4b and the interactive browser)
comes *before* any aggregate. An event-aligned average is only as good as the event annotation
underneath it, and the annotation is the part most likely to be wrong — see the backtracking
caveat in the detection cell.

In [ ]:
# --- Step 1: per-animal excursion threshold ------------------------------------------------
# H3 used two fixed constants for every animal (EXC_THR = 2.5 vu, BASE_MAX = 1.0 vu). Animals
# differ in how tightly they hold their route, so a fixed cutoff calls a different fraction of
# each animal's trials "off-route" -- JB2's route scatter is not JB4's.
#
# Derive the cutoff per animal from that animal's OWN pooled per-bin deviation, exactly the way
# the error threshold is derived: a FIXED threshold pooled over all sessions, deliberately NOT a
# per-session percentile (that would manufacture a constant event rate in every session by
# construction, which is the same trap the ERR_PCTL cell calls out).
#
# CALIBRATION. The percentile is not a free choice. Picking a round number (95) put the cutoff at
# 7.2-7.9 vu, within a whisker of the 9 vu half-width -- a detector that only fires when the animal
# is against the wall. Instead, measure which percentile the H3 constant 2.5 vu actually occupies
# in each animal's deviation distribution, and use the average of those. The threshold then adapts
# to each animal while staying anchored to the criterion H3 was built and sanity-checked on.
BASE_FRAC = 1.0 / 2.5      # preserves the original BASE_MAX / EXC_THR ratio exactly

def _late_ref(keys, k, means):
    """Leave-one-out late template: mean of the animal's last 3 sessions, excluding k."""
    return np.mean([means[j] for j in keys[-3:] if j != k], axis=0)

DEV_BINS = {}          # animal -> pooled |X - ref| over every trial and every bin
for a in TRAJ_ANIMALS:
    keys = sorted([k for k in TRAJ if k.startswith(a + "|")], key=lambda k: k.split("|")[1:])
    means = {k: TRAJ[k]['X'].mean(0) for k in keys}
    DEV_BINS[a] = np.concatenate(
        [np.abs(TRAJ[k]['X'] - _late_ref(keys, k, means)).ravel() for k in keys])

PCTL_OF_FIXED = {a: 100.0 * float((DEV_BINS[a] < EXC_THR).mean()) for a in TRAJ_ANIMALS}
EXC_PCTL = float(np.mean([PCTL_OF_FIXED[a] for a in TRAJ_ANIMALS]))

EXC_THR_A  = {a: float(np.percentile(DEV_BINS[a], EXC_PCTL)) for a in TRAJ_ANIMALS}
BASE_MAX_A = {a: BASE_FRAC * EXC_THR_A[a] for a in TRAJ_ANIMALS}

print("where the H3 constant %.1f vu sits in each animal's pooled |deviation|:" % EXC_THR)
for a in TRAJ_ANIMALS:
    print("   %-5s %5.1fth percentile" % (a, PCTL_OF_FIXED[a]))
print("-> EXC_PCTL = %.1f (their mean)\n" % EXC_PCTL)

print("per-animal excursion thresholds:")
print("   %-5s %10s %10s %10s   %s" % ('animal', 'EXC_THR', 'BASE_MAX', '(cm)', 'vs fixed 2.5 vu'))
for a in TRAJ_ANIMALS:
    print("   %-5s %8.2f vu %7.2f vu %8.2f cm   %+.2f vu"
          % (a, EXC_THR_A[a], BASE_MAX_A[a], EXC_THR_A[a] * CM_PER_VU, EXC_THR_A[a] - EXC_THR))

# What the new threshold does to the H3 event count, on the SAME spatial detector, so the change
# is attributable to the threshold alone and not to the move into the time domain.
def _detect_spatial(thr_a, base_a):
    rows = []
    for a in TRAJ_ANIMALS:
        keys = sorted([k for k in TRAJ if k.startswith(a + "|")], key=lambda k: k.split("|")[1:])
        means = {k: TRAJ[k]['X'].mean(0) for k in keys}
        for k in keys:
            ref = _late_ref(keys, k, means)
            for i in range(len(TRAJ[k]['X'])):
                d = np.abs(TRAJ[k]['X'][i] - ref)
                rr = [r for r in _runs(d > thr_a[a])
                      if r[0] >= EDGE_BINS and r[1] <= N_BIN - 1 - EDGE_BINS]
                if len(rr) != 1:
                    continue
                s, e = rr[0]
                if not (EXC_WIDTH[0] <= e - s + 1 <= EXC_WIDTH[1]):
                    continue
                outside = np.concatenate([d[:max(s - 1, 0)], d[min(e + 2, N_BIN):]])
                if len(outside) < 10 or outside.mean() > base_a[a]:
                    continue
                rows.append(a)
    return pd.Series(rows).value_counts() if rows else pd.Series(dtype=int)

_new = _detect_spatial(EXC_THR_A, BASE_MAX_A)
_old = exc.groupby('animal').size()
print("\nspatial-detector event counts, fixed 2.5 vu vs per-animal threshold:")
for a in TRAJ_ANIMALS:
    print("   %-5s  fixed %4d   per-animal %4d" % (a, int(_old.get(a, 0)), int(_new.get(a, 0))))
print("   %-5s  fixed %4d   per-animal %4d" % ('TOTAL', int(_old.sum()), int(_new.sum())))
print("\nJB2 is the binding constraint on every per-animal panel in H4. If its count falls far")
print("below ~70 events, lower EXC_PCTL by hand and re-run from here.")

### A time-resolved cache

`prelim_cache_sessions.pkl` is nearly what H4 needs, but not quite: it is a **lazy** cache holding
only the handful of sessions the H3 example figures happened to touch, and its per-trial tuple
`(x, y, dx, dy)` carries **no time axis at all**.

Rebuilding it means one pass over `Y:\virmen`, which is the slow part of this whole section.

Two departures from the H3 cache, both deliberate:

* **One `.npz` per session** in `prelim_cache_time/`, instead of a single pickle. The full set is
  ~200 MB; the H3 cache re-pickles the entire growing dict on *every* session it adds, which is
  quadratic and would rewrite hundreds of MB per session here. Per-session files also let the
  browser load only the session being looked at, and a crash costs one session rather than all of them.
* **`t` is stored relative to each trial's start**, not as absolute session time. Session time
  reaches a few thousand seconds, and at float32 that would quantise the ~18 ms sample interval;
  trial-relative time never exceeds `MAX_DUR_S`.

`dt == 0` samples exist in this data. `cumsum` handles them without complaint, but nothing below
ever divides by `dt` — the recorded `dx`/`dy` velocities are used instead of differentiating.

**One correction is applied on the way in.** `load_vr` flips `x` (`df['x'] = -df['x']`, following
`wideLinearTrack.py:54`) but leaves `dx` untouched, so as loaded the two disagree about which way
is right. Measured directly on raw `sessionData`, `diff(x_raw) / (dx*dt) = +1.000` — `dx` tracks
the *unflipped* `x`. The cache therefore stores `-dx`, so that lateral velocity is the rate of
change of the same `x` every figure plots. Left uncorrected it is not a cosmetic problem: the
phase portrait traverses its loop backwards and `dx` appears to oppose the deviation it is
producing. A QC assertion below pins the sign so it cannot regress silently.

In [ ]:
# --- Step 2: per-sample cache WITH a time axis ---------------------------------------------
TIME_DIR = os.path.join(CACHE_DIR, 'prelim_cache_time')
TIME_VER = 'v2'        # v2: dx sign corrected to match the flipped x (see below)
os.makedirs(TIME_DIR, exist_ok=True)

def _time_path(key):
    return os.path.join(TIME_DIR, key.replace("|", "_") + "_" + TIME_VER + ".npz")

# Drop caches from an earlier version rather than leaving ~130 MB of stale data behind.
for _f in os.listdir(TIME_DIR):
    if _f.endswith('.npz') and ('_' + TIME_VER + '.npz') not in _f:
        os.remove(os.path.join(TIME_DIR, _f))

def build_session_time(animal, date, sess, overwrite=False, retries=3):
    r"""Read one session off the Y: share and write per-sample x, y, dx, dy and time.

    Retries because Y:\virmen is an SMB share that drops reads intermittently: a full pass has
    failed partway with FileNotFoundError on a path that exists and reads fine seconds later.
    A couple of retries costs nothing and saves losing a 7-minute pass to a blip.
    """
    import time as _time
    key = "%s|%s|%s" % (animal, date, sess)
    p = _time_path(key)
    if os.path.exists(p) and not overwrite:
        return False
    for _att in range(retries):
        try:
            df, _meta = load_vr(animal, date, sess)
            break
        except (FileNotFoundError, OSError) as e:
            if _att == retries - 1:
                raise
            print("   transient read failure on %s (%s) - retrying" % (key, type(e).__name__))
            _time.sleep(2.0)
    x  = df['x'].to_numpy();  y  = df['y'].to_numpy()
    # SIGN. load_vr flips x (`df['x'] = -df['x']`, per wideLinearTrack.py:54) but leaves dx
    # alone, so as loaded they disagree: measured on raw sessionData, diff(x_raw)/(dx*dt) =
    # +1.000 exactly, i.e. dx tracks the UNflipped x. Negating it here is what makes "lateral
    # velocity" mean the rate of change of the x that every H3/H4 figure plots. Without this
    # the phase portrait traverses backwards and dx appears to oppose the deviation it causes.
    vx = -df['dx'].to_numpy(); vy = df['dy'].to_numpy()
    tt = df['t'].to_numpy();  tr = df['trial'].to_numpy()

    edges = np.flatnonzero(np.diff(tr) != 0) + 1
    st = np.concatenate([[0], edges]); sp = np.concatenate([edges, [len(tr)]])

    xs, ys, vxs, vys, ts, tid, off = [], [], [], [], [], [], [0]
    for a, b in zip(st, sp):
        if b - a < 3:
            continue
        if (tt[b - 1] - tt[a]) > MAX_DUR_S:       # same disengagement cap as the trial table
            continue
        xs.append(x[a:b]); ys.append(y[a:b]); vxs.append(vx[a:b]); vys.append(vy[a:b])
        ts.append(tt[a:b] - tt[a])                # trial-relative, so float32 keeps ~18 ms resolution
        tid.append(int(tr[a])); off.append(off[-1] + (b - a))
    if not tid:
        return False

    y_goal = float(y.max())
    y_start_med = float(np.median([y[a] for a, b in zip(st, sp) if b - a >= 3]))
    be = np.linspace(y_start_med + 2.0, y_goal - 2.0, N_BIN + 1)
    np.savez_compressed(
        p,
        x=np.concatenate(xs).astype(np.float32),  y=np.concatenate(ys).astype(np.float32),
        vx=np.concatenate(vxs).astype(np.float32), vy=np.concatenate(vys).astype(np.float32),
        t=np.concatenate(ts).astype(np.float32),
        tid=np.asarray(tid, dtype=np.int32), off=np.asarray(off, dtype=np.int32),
        centres=0.5 * (be[:-1] + be[1:]), y_goal=np.asarray(y_goal))
    return True

def session_centres(key):
    """Just the y-bin centres and y_goal, without materialising the whole session."""
    p = _time_path(key)
    if not os.path.exists(p):
        build_session_time(*key.split("|"))
    with np.load(p) as z:
        return z['centres'], float(z['y_goal'])

_TIME_CACHE = {}          # key -> (trials, centres, y_goal); in-memory, one session at a time

def session_time(key):
    """{trial_id: (x, y, dx, dy, t)} for one session, plus y-bin centres and y_goal."""
    if key in _TIME_CACHE:
        return _TIME_CACHE[key]
    p = _time_path(key)
    if not os.path.exists(p):
        build_session_time(*key.split("|"))
    with np.load(p) as z:
        # Materialise each signal ONCE. Indexing an NpzFile re-decompresses the whole array on
        # every access, and a slice of it is a VIEW that keeps that copy alive -- doing it per
        # trial retains one full-session array per trial per signal (hundreds of MB a session).
        x, y, vx, vy, t = z['x'], z['y'], z['vx'], z['vy'], z['t']
        off, tid = z['off'], z['tid']
        centres, y_goal = z['centres'], float(z['y_goal'])
    trials = {int(tid[i]): (x[off[i]:off[i + 1]], y[off[i]:off[i + 1]],
                            vx[off[i]:off[i + 1]], vy[off[i]:off[i + 1]],
                            t[off[i]:off[i + 1]])
              for i in range(len(tid))}
    out = (trials, centres, y_goal)
    if len(_TIME_CACHE) > 6:                      # ~2.5 MB/session; keep the browser responsive
        _TIME_CACHE.clear()
    _TIME_CACHE[key] = out
    return out

TIME_KEYS = sorted(TRAJ.keys(), key=lambda k: (k.split("|")[0], k.split("|")[1:]))
_missing = [k for k in TIME_KEYS if not os.path.exists(_time_path(k))]
print("time cache: %d of %d sessions present, %d to build" %
      (len(TIME_KEYS) - len(_missing), len(TIME_KEYS), len(_missing)))
if _missing:
    print("building (one pass over the share -- this is the slow step) ...")
    for i, k in enumerate(_missing):
        try:
            build_session_time(*k.split("|"))
        except Exception as e:
            print("  FAILED %s: %s" % (k, e))
        if (i + 1) % 10 == 0:
            print("  ... %d/%d" % (i + 1, len(_missing)))
    print("done")

# A session whose read never succeeded is dropped from the analysis set rather than being left
# to raise later from whichever cell happens to touch it first.
_dropped = [k for k in TIME_KEYS if not os.path.exists(_time_path(k))]
TIME_KEYS = [k for k in TIME_KEYS if os.path.exists(_time_path(k))]
TIME_OK = set(TIME_KEYS)
if _dropped:
    print("WARNING: %d session(s) have no time cache and are excluded from H4:" % len(_dropped))
    for k in _dropped:
        print("   " + k)
_mb = sum(os.path.getsize(_time_path(k)) for k in TIME_KEYS)
print("on disk: %.0f MB across %d sessions" % (_mb / 1e6, len(TIME_KEYS)))

# QC: x and dx must now agree, and t must reproduce the trial duration in the trial table.
# The dx sign is easy to regress silently and inverts the phase portrait when it does.
_qc = TIME_KEYS[0]
_tr, _c, _g = session_time(_qc)
_rx, _rt = [], []
_ttab = trials.set_index(['animal', 'date', 'session', 'trial'])['dur_s']
for _tid, (_x, _y, _vx, _vy, _t) in list(_tr.items())[:40]:
    _dt = np.diff(_t)
    _m = (_dt > 0) & (np.abs(_vx[:-1]) > 0.5)
    if _m.sum() > 20:
        _rx.append(np.median(np.diff(_x)[_m] / (_vx[:-1][_m] * _dt[_m])))
    _a, _d, _s = _qc.split("|")
    try:
        _rt.append(float(_t[-1]) - float(_ttab.loc[(_a, int(_d), _s, _tid)]))
    except KeyError:
        pass
print("QC %s: median diff(x)/(dx*dt) = %+.3f  (must be +1: x and dx share a sign)"
      % (_qc, float(np.median(_rx))))
print("QC cached t vs trials['dur_s']: max |difference| = %.4f s over %d trials"
      % (float(np.max(np.abs(_rt))) if _rt else np.nan, len(_rt)))
assert abs(float(np.median(_rx)) - 1.0) < 0.05, 'dx sign disagrees with x -- see the note above'

### Re-detecting the excursions in time

The H3 detector runs on `TRAJ['X']`, the 40-bin spatial resample, which is **monotonic in `y` by
construction** — each bin appears exactly once, in order. The per-sample series is not. Three
consequences, and they are the reason the verification figures come next rather than the averages:

* The route has to be evaluated at each sample's own `y` (`np.interp` onto `centres`) rather than
  read off a bin index.
* **A trial that backtracks revisits the same `y`.** One spatial excursion can therefore break
  into several temporal runs, or two spatial excursions can merge into one. The count of trials
  whose event structure changes is printed below — that number is the honest measure of how much
  the two detectors disagree, and `max_backtrack` is kept as a column so it can be conditioned on.
* At ~56 Hz a single noisy sample can drop below threshold mid-excursion and split a run in two.
  Runs separated by less than `MERGE_GAP_S` are merged before the "exactly one run" test, which is
  a sample-rate artefact fix, not a change to the criterion.

The width criterion moves from bins to the **`y`-extent** the excursion covers, converted through
the bin width so the admissible range is the same stretch of track as before.

In [ ]:
# --- Step 3: detect excursions on the per-sample series ------------------------------------
MERGE_GAP_S = 0.15     # sub-threshold gaps shorter than this are sampling noise, not a return
GUARD_S     = 0.10     # excluded either side of the run when measuring the on-route baseline

def _merge_runs(rr, t, gap_s):
    """Merge runs separated by a shorter sub-threshold gap than gap_s."""
    if not rr:
        return rr
    out = [list(rr[0])]
    for s, e in rr[1:]:
        if t[s] - t[out[-1][1]] <= gap_s:
            out[-1][1] = e
        else:
            out.append([s, e])
    return [tuple(r) for r in out]

def trial_dev(key, trial_id, ref, centres):
    """Per-sample deviation of one trial from the route, plus the raw samples."""
    trials, _c, _g = session_time(key)
    xx, yy, vx, vy, tt = trials[int(trial_id)]
    return xx, yy, vx, vy, tt, xx - np.interp(yy, centres, ref)

REF_T = {}             # key -> (leave-one-out late template, y-bin centres)
for a in TRAJ_ANIMALS:
    _keys = sorted([k for k in TRAJ if k.startswith(a + "|")], key=lambda k: k.split("|")[1:])
    _means = {k: TRAJ[k]['X'].mean(0) for k in _keys}
    # The late template is still the mean of the last 3 sessions of the FULL set, so a session
    # missing a time cache cannot silently change the reference the others are scored against.
    for k in _keys:
        if k in TIME_OK:
            REF_T[k] = (_late_ref(_keys, k, _means), session_centres(k)[0])

exc_t_rows, _frag = [], 0
for a in TRAJ_ANIMALS:
    keys = sorted([k for k in TRAJ if k.startswith(a + "|")], key=lambda k: k.split("|")[1:])
    thr, base_max = EXC_THR_A[a], BASE_MAX_A[a]
    for si, k in enumerate(keys):
        if k not in REF_T:          # no time cache for this session; see the build cell
            continue
        ref, centres = REF_T[k]
        bw = (centres[-1] - centres[0]) / (N_BIN - 1)          # vu per bin
        y_lo_ok, y_hi_ok = centres[EDGE_BINS], centres[N_BIN - 1 - EDGE_BINS]
        v = TRAJ[k]
        trials, _c, _g = session_time(k)
        for i in range(len(v['X'])):
            tid = int(v['tr'][i])
            if tid not in trials:
                continue
            xx, yy, vx, vy, tt, d_signed = trial_dev(k, tid, ref, centres)
            d = np.abs(d_signed)
            rr_raw = _runs(d > thr)
            rr = _merge_runs(rr_raw, tt, MERGE_GAP_S)
            if len(rr_raw) != len(rr):
                _frag += 1
            rr = [r for r in rr if yy[r[0]] >= y_lo_ok and yy[r[1]] <= y_hi_ok]
            if len(rr) != 1:
                continue
            s, e = rr[0]
            yspan = float(yy[s:e + 1].max() - yy[s:e + 1].min())
            if not (EXC_WIDTH[0] * bw <= yspan <= EXC_WIDTH[1] * bw):
                continue
            g = max(int(round(GUARD_S / max(np.median(np.diff(tt)), 1e-3))), 1)
            outside = np.concatenate([d[:max(s - g, 0)], d[min(e + g + 1, len(d)):]])
            if len(outside) < 20:
                continue
            bs = float(outside.mean())
            if bs > base_max:
                continue
            pk = s + int(np.argmax(d[s:e + 1]))
            exc_t_rows.append(dict(
                animal=a, key=k, date=k.split("|")[1], session=k.split("|")[2], sess_i=si,
                trial=tid, rewarded=bool(v['rew'][i]), max_backtrack=float(v['bt'][i]),
                i_onset=int(s), i_peak=int(pk), i_offset=int(e),
                t_onset=float(tt[s]), t_peak=float(tt[pk]), t_offset=float(tt[e]),
                t_out=float(tt[pk] - tt[s]), t_back=float(tt[e] - tt[pk]),
                t_total=float(tt[e] - tt[s]), dur_s=float(tt[-1]),
                y_onset=float(yy[s]), y_peak=float(yy[pk]), y_offset=float(yy[e]),
                yspan=yspan, peak=float(d[pk]), base=bs, prominence=float(d[pk] - bs),
                signed_peak=float(d_signed[pk]),
            ))
exc_t = pd.DataFrame(exc_t_rows)

print("temporal detector: %d events in %d trials scored (%.1f%%)"
      % (len(exc_t), len(dev), 100.0 * len(exc_t) / len(dev)))
print(exc_t.groupby('animal').agg(n=('trial', 'size'), median_peak=('peak', 'median'),
                                  median_t_out=('t_out', 'median'),
                                  median_t_back=('t_back', 'median'),
                                  median_t_total=('t_total', 'median'),
                                  frac_rewarded=('rewarded', 'mean')).round(3).to_string())

# --- agreement with the spatial detector ---------------------------------------------------
_sp = set(zip(exc['key'], exc['trial']))
_tm = set(zip(exc_t['key'], exc_t['trial']))
print("\nagreement with the H3 spatial detector (note: thresholds also differ, so this is a")
print("combined effect of the threshold change AND the move into the time domain)")
print("   both detectors      %4d" % len(_sp & _tm))
print("   spatial only        %4d   (lost in time: backtracking, merging, or width in vu)" % len(_sp - _tm))
print("   temporal only       %4d   (found in time: split spatial bins, or the new threshold)" % len(_tm - _sp))
print("   runs merged across a sub-threshold gap: %d trials" % _frag)
_bt = exc_t['max_backtrack'] > MAX_BACKTRACK_VU
print("\n%d of %d temporal events (%.1f%%) are on trials that backtrack more than %.0f vu."
      % (_bt.sum(), len(exc_t), 100.0 * _bt.mean(), MAX_BACKTRACK_VU))
print("They are FLAGGED, not dropped -- check them in the browser before deciding.")

### Shared drawing helpers

Three things below draw the same trial at different sizes — the verification figure, the contact
sheet and the interactive browser — so the drawing is written once here.

`plot_jg.py` already has a `colored_line` (line 214) and a `draw_lineartrack` (line 895), but
`plot_jg` imports `functions as fc` and `mouse_imaging.session`, neither of which resolves on this
machine. That is why this notebook is standalone, and why these are transcribed rather than imported.

`DIV_CMAP` is **not** reused for the time ramp: it is a diverging map and passes through near-white
at its midpoint, which would make the middle of every trial invisible against `SURFACE`. `TIME_CMAP`
runs blue through purple to dark red at roughly constant lightness instead.

The **isochrone dots** are placed at a fixed time interval, so their spacing along the path *is*
speed — the dots bunch up wherever the animal slows. That puts a velocity read-out inside the
spatial plot, which matters because slowing during a correction is one of the things h4g tests.

In [ ]:
# --- Step 4: shared drawing helpers --------------------------------------------------------
from matplotlib.collections import LineCollection

# Blue -> purple -> dark red. Avoids the near-white midpoint of DIV_CMAP, which would erase the
# middle of every trial against the SURFACE background.
TIME_CMAP = LinearSegmentedColormap.from_list(
    'labtime', ['#1c5cab', '#4a63b5', '#7a56a8', '#a34a80', '#c94553', '#8f1f1e'])

ANN_ON_ROUTE = MUTED        # on route
ANN_OUT      = CRITICAL     # leaving the route: onset -> peak
ANN_BACK     = BLUE         # correcting back:   peak  -> offset
EVENT_SHADE  = '#f2b3b2'    # same shade the H3 figures use for the off-route band

def colored_line(ax, x, y, c=None, cmap=None, colors=None, lw=1.3,
                 vmin=None, vmax=None, zorder=3):
    """Per-segment coloured line. Mirrors plot_jg.py:214, which is unimportable here."""
    pts = np.asarray([x, y]).T.reshape(-1, 1, 2)
    segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
    if colors is not None:
        lc = LineCollection(segs, colors=list(colors)[:-1], linewidths=lw, zorder=zorder)
    else:
        lc = LineCollection(segs, cmap=cmap, linewidths=lw, zorder=zorder)
        lc.set_array(np.asarray(c)[:-1])
        if vmin is not None:
            lc.set_clim(vmin, vmax)
    ax.add_collection(lc)
    return lc

def annotation_colors(n, ev=None):
    """Per-sample colours for colour_by='annotation'."""
    cols = [ANN_ON_ROUTE] * n
    if ev is not None:
        for i in range(int(ev['i_onset']), min(int(ev['i_peak']) + 1, n)):
            cols[i] = ANN_OUT
        for i in range(int(ev['i_peak']), min(int(ev['i_offset']) + 1, n)):
            cols[i] = ANN_BACK
    return cols

def draw_trial_path(ax, xx, yy, tt, ev=None, colour_by='time', dots_every_s=0.25,
                    y_goal=None, lw=1.3, t_max=None, marks=True):
    """One trial's x-y path in maze coordinates, with the track outline."""
    yg = float(y_goal if y_goal is not None else np.max(yy))
    for w in (-10, 10):                                   # track walls, per plot_jg.draw_lineartrack
        ax.axvline(w, color=BASELINE, lw=0.8, zorder=0)
    ax.axvline(0, color=GRID, lw=0.8, zorder=0)
    ax.scatter([0], [yg], marker='v', s=26, color=MUTED, zorder=1)   # reward tower

    if colour_by == 'time':
        lc = colored_line(ax, xx, yy, c=tt, cmap=TIME_CMAP, lw=lw,
                          vmin=0.0, vmax=float(t_max if t_max is not None else tt[-1]))
    else:
        lc = colored_line(ax, xx, yy, colors=annotation_colors(len(xx), ev), lw=lw)

    if dots_every_s:
        tg = np.arange(0.0, float(tt[-1]), dots_every_s)
        ax.scatter(np.interp(tg, tt, xx), np.interp(tg, tt, yy),
                   s=2.0, color=INK, alpha=0.45, zorder=4, linewidths=0)
    if marks and ev is not None:
        for i, mk, fc in ((ev['i_onset'], 'o', SURFACE), (ev['i_peak'], 'o', CRITICAL),
                          (ev['i_offset'], 's', SURFACE)):
            i = int(i)
            ax.scatter([xx[i]], [yy[i]], marker=mk, s=28, facecolor=fc,
                       edgecolor=INK, linewidths=0.9, zorder=6)
    ax.set_xlim(-11, 11); ax.set_ylim(0, yg * 1.02); ax.grid(False)
    return lc

TS_LABELS = ['y (vu)', 'x (vu)', 'dev (vu)', 'dx (vu/s)', 'dy (vu/s)']

def draw_trial_timeseries(axs, xx, yy, vx, vy, tt, d_signed, route_x, ev=None, thr=None):
    """Stacked y / x / deviation / dx / dy against time, event window shaded in every row."""
    series = [(yy, INK), (xx, INK), (d_signed, INK), (vx, INK_2), (vy, INK_2)]
    for ax, (v, col), lab in zip(axs, series, TS_LABELS):
        if ev is not None:
            ax.axvspan(ev['t_onset'], ev['t_offset'], color=EVENT_SHADE, alpha=0.45, lw=0, zorder=0)
            ax.axvline(ev['t_peak'], color=CRITICAL, lw=0.8, ls=':', zorder=1)
        ax.plot(tt, v, color=col, lw=1.0, zorder=3)
        ax.set_ylabel(lab, fontsize=7)
        ax.tick_params(labelsize=7)
    axs[1].plot(tt, route_x, color=MUTED, lw=1.2, zorder=2)      # the usual route, in time
    for ax in (axs[2], axs[3], axs[4]):
        ax.axhline(0, color=BASELINE, lw=0.8, zorder=1)
    if thr is not None:
        for s in (+1, -1):
            axs[2].axhline(s * thr, color=CRITICAL, lw=0.8, ls='--', zorder=2)
    axs[-1].set_xlabel('time in trial (s)', fontsize=7)

def event_of(key, trial):
    """The exc_t row for one trial as a plain dict, or None."""
    m = exc_t[(exc_t['key'] == key) & (exc_t['trial'] == int(trial))]
    return None if m.empty else m.iloc[0].to_dict()

### h4a — single-trial verification

**Read this before anything below it.** Every aggregate in H4 assumes the event annotation is
right; this is where that assumption is checked.

The same six trials the H3 figures use, ranked by prominence, so the spatial view already saved in
`h3_transient_error_corrections` and this time view show the same animals on the same trials.

Each column is one trial, drawn twice as a path — once coloured by **time**, once by the
detector's **annotation** — over the five time series the detector actually runs on. What to look
for: the red (out) limb should be the animal leaving its route and the blue (back) limb should be
it returning, `dx` should change sign near the peak, and the shaded window should start and end
where the deviation trace crosses the dashed threshold. If the annotation instead straddles a
backtrack, or the "correction" is really the animal wandering back down the track, that shows up
here and nowhere else.

In [ ]:
# --- h4a: single-trial verification, read FIRST --------------------------------------------
N_EX_PER_ANIMAL = 2
picks_t = []
for a in TRAJ_ANIMALS:
    picks_t += list(exc_t[exc_t['animal'] == a]
                    .nlargest(N_EX_PER_ANIMAL, 'prominence').itertuples())

T_MAX = max(float(r.dur_s) for r in picks_t)      # shared colour scale, so panels compare

fig = plt.figure(figsize=(2.55 * len(picks_t), 13.2))
gs = fig.add_gridspec(7, len(picks_t), height_ratios=[2.7, 2.7, 1, 1, 1, 1, 1],
                      hspace=0.30, wspace=0.34)
lc_ref = None
for j, r in enumerate(picks_t):
    ev = r._asdict()
    ref, centres = REF_T[r.key]
    xx, yy, vx, vy, tt, d = trial_dev(r.key, r.trial, ref, centres)
    route_x = np.interp(yy, centres, ref)
    y_goal = session_time(r.key)[2]
    an, dd_, ss_ = r.key.split("|")

    ax_t = fig.add_subplot(gs[0, j])
    lc_ref = draw_trial_path(ax_t, xx, yy, tt, ev, 'time', y_goal=y_goal, t_max=T_MAX)
    ax_t.set_title("%s %s tr%d\npeak %.1f vu %s   %.1f s\nt_out %.2f s   t_back %.2f s%s"
                   % (an, dd_, r.trial, r.peak, 'right' if r.signed_peak > 0 else 'left',
                      r.dur_s, r.t_out, r.t_back, "" if r.rewarded else "\nNO REWARD"),
                   loc='left', fontsize=7)

    ax_a = fig.add_subplot(gs[1, j], sharex=ax_t, sharey=ax_t)
    draw_trial_path(ax_a, xx, yy, tt, ev, 'annotation', y_goal=y_goal)

    axs = []
    for k in range(5):
        ax = fig.add_subplot(gs[2 + k, j], sharex=axs[0] if axs else None)
        axs.append(ax)
        if k < 4:
            ax.tick_params(labelbottom=False)
    draw_trial_timeseries(axs, xx, yy, vx, vy, tt, d, route_x, ev, thr=EXC_THR_A[an])

    if j == 0:
        ax_t.set_ylabel('y (vu) - coloured by time'); ax_a.set_ylabel('y (vu) - by annotation')
    else:
        for ax in (ax_t, ax_a):
            ax.tick_params(labelleft=False)
    ax_a.set_xlabel('x (vu)', fontsize=7)

cb = fig.colorbar(lc_ref, ax=fig.get_axes(), location='right', fraction=0.012, pad=0.01)
cb.set_label('time in trial (s)', fontsize=8); cb.ax.tick_params(labelsize=7)

from matplotlib.lines import Line2D
fig.legend(handles=[Line2D([], [], color=ANN_ON_ROUTE, lw=2, label='on route'),
                    Line2D([], [], color=ANN_OUT, lw=2, label='leaving (onset - peak)'),
                    Line2D([], [], color=ANN_BACK, lw=2, label='correcting (peak - offset)'),
                    Line2D([], [], color=INK, marker='.', ls='none', label='0.25 s apart')],
           loc='upper right', bbox_to_anchor=(0.995, 0.995), ncol=4, fontsize=8)
fig.suptitle('h4a - single-trial verification: does the detector mark a real correction?',
             x=0.01, ha='left', fontsize=11, color=INK)
plt.show()

### h4b — whole-session contact sheet

h4a can only confirm events that were already found. It says nothing about **false negatives** —
trials that contain an obvious correction the detector walked straight past — and those are the
failure mode that silently biases every duration statistic downstream.

So: every trial of one session as a thumbnail, ordered by `dev_rms` rather than by trial number,
which puts the threshold boundary in the middle of the sheet with the ambiguous cases adjacent to
each other. Outlines mark what the two detectors said:

* **red** — the temporal detector called a correction
* **orange** — the H3 spatial detector called one and the temporal detector did not (disagreement)
* **grey** — neither

Scan for red-worthy trials still outlined grey. Generated for an early and a late session, because
a threshold that looks right on a well-trained animal often over-fires on an early one.

In [ ]:
# --- h4b: whole-session contact sheet ------------------------------------------------------
CONTACT_COLS, CONTACT_ROWS, CONTACT_MAX_PAGES = 10, 8, 2

_DEV_RMS = {(r.key, r.trial): r.dev_rms for r in dev.itertuples()}
_SPATIAL = set(zip(exc['key'], exc['trial']))
_TEMPORAL = set(zip(exc_t['key'], exc_t['trial']))

def contact_sheet(key, cols=CONTACT_COLS, rows=CONTACT_ROWS, max_pages=CONTACT_MAX_PAGES,
                  step=3):
    """Every trial of one session as a thumbnail, ordered by dev_rms."""
    an, dd_, ss_ = key.split("|")
    ref, centres = REF_T[key]
    trials, _c, y_goal = session_time(key)
    ids = sorted([t for t in trials if (key, t) in _DEV_RMS], key=lambda t: _DEV_RMS[(key, t)])
    per = cols * rows
    pages = min(int(np.ceil(len(ids) / per)), max_pages)
    # Robust colour ceiling: one 40 s outlier would otherwise squash every ordinary trial into
    # the blue end of the ramp and throw away the whole point of colouring by time.
    t_max = float(np.percentile([float(trials[t][4][-1]) for t in ids], 90))

    for pg in range(pages):
        chunk = ids[pg * per:(pg + 1) * per]
        fig, axes = plt.subplots(rows, cols, figsize=(1.15 * cols, 1.65 * rows))
        axes = np.atleast_1d(axes).ravel()
        for ax in axes:
            ax.set_axis_off()
        for ax, tid in zip(axes, chunk):
            ax.set_axis_on(); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
            xx, yy, _vx, _vy, tt = trials[tid]
            ev = event_of(key, tid)
            colored_line(ax, xx[::step], yy[::step], c=tt[::step], cmap=TIME_CMAP,
                         lw=0.8, vmin=0.0, vmax=t_max)
            ax.plot(ref, centres, color=GRID, lw=0.7, zorder=1)
            if ev is not None:
                ax.axhspan(min(ev['y_onset'], ev['y_offset']), max(ev['y_onset'], ev['y_offset']),
                           color=EVENT_SHADE, alpha=0.40, lw=0, zorder=0)
            edge = (CRITICAL if (key, tid) in _TEMPORAL else
                    ORANGE if (key, tid) in _SPATIAL else BASELINE)
            # The global rcParams hide the top and right spines, which would leave each verdict
            # drawn as an L rather than a box. Turn all four back on for the tiles.
            for sp in ax.spines.values():
                sp.set_visible(True)
                sp.set_color(edge)
                sp.set_linewidth(1.8 if edge != BASELINE else 0.6)
            ax.set_xlim(-11, 11); ax.set_ylim(0, y_goal * 1.02)
            ax.set_title("tr%d  d=%.1f" % (tid, _DEV_RMS[(key, tid)]), fontsize=5.5, pad=1.5,
                         color=INK if edge == BASELINE else edge)
        fig.suptitle('h4b - %s %s %s, page %d/%d - every trial, ordered by dev_rms\n'
                     'red = temporal event   orange = spatial only   grey = neither'
                     % (an, dd_, ss_, pg + 1, pages), x=0.01, ha='left', fontsize=9, color=INK)
        fig.tight_layout(rect=(0, 0, 1, 0.945))
        plt.show()
    print("%s: %d trials, %d shown over %d page(s)"
          % (key, len(ids), min(len(ids), pages * per), pages))

_rich = exc_t['animal'].value_counts().index[0]
_ck = sorted([k for k in TIME_OK if k.startswith(_rich + "|")], key=lambda k: k.split("|")[1:])
CONTACT_KEYS = [_ck[0], _ck[-1]]          # one early, one late session of the richest animal
for _k in CONTACT_KEYS:
    contact_sheet(_k)

### An interactive browser

A slider over one session, ordered by `dev_rms` so that stepping walks you across the threshold
boundary rather than through the session chronologically — the ambiguous trials arrive together
instead of being scattered through a few hundred ordinary ones.

This cell produces **no saved figure**; it is a tool for looking, not an output.

**Use `show_trial()` — it needs no widgets.** ipywidgets loads its JavaScript from a CDN, and on
this network that fetch fails (VS Code logs `Widget Error: Failed to access CDN
https://unpkg.com`), so the slider version cannot render. `show_trial()` is a plain function that
draws one trial with matplotlib; you step through by editing a number and re-running the cell:

```python
show_trial(0)                          # tidiest trial of the session, by dev_rms
show_trial(-1)                         # worst
show_trial(-1, colour_by='annotation') # what the detector called it
show_trial(5, only_events=True)        # step through detected corrections only
show_trial(0, key='JB2|260717|session_1')
```

Ranking is by `dev_rms` rather than trial number, so walking the index takes you across the
threshold boundary and the ambiguous cases arrive together. `browse_trials()` is still there for
the slider if widgets ever work.

**Neither is auto-run.** A "Run All" should not build a widget tree, and the cell deliberately
does not switch the matplotlib backend — doing that inside a live kernel, after figures already
exist, is a good way to destabilise it. If you want pan/zoom, type `%matplotlib widget` in its
own cell first, by hand.

One constraint worth keeping in mind if you edit this cell: `regenerate_figures.py` executes
every code cell with `compile()`, so a bare IPython magic in the notebook is a `SyntaxError`
there. It now strips magic lines defensively, but only for cells that would not otherwise
compile, so nothing valid is ever altered.

In [ ]:
# --- trial browser: a no-widget version, and a widget version ------------------------------
# NOTE: no `%matplotlib widget` magic anywhere in this cell. regenerate_figures.py runs
# compile() over every code cell, and a magic line is not valid Python -- it would raise
# SyntaxError and abort the entire headless regeneration.
import sys

def _browse_ids(key, only_events=False):
    """Trial ids for one session, ordered by dev_rms (ascending)."""
    trials, _c, _g = session_time(key)
    ids = sorted([t for t in trials if (key, t) in _DEV_RMS], key=lambda t: _DEV_RMS[(key, t)])
    if only_events:
        ids = [t for t in ids if (key, t) in _TEMPORAL]
    return ids

def show_trial(rank=0, key=None, colour_by='time', only_events=False, figsize=(11.0, 6.4)):
    """Draw ONE trial: path + stacked time series. No widgets, no JavaScript.

    `rank` indexes trials ordered by dev_rms, so rank=0 is the tidiest trial of the session and
    rank=-1 the worst. Negative indices work. Step through by editing the number and re-running:
        show_trial(0)      show_trial(-1)      show_trial(12, only_events=True)
    """
    key = key or CONTACT_KEYS[-1]
    ids = _browse_ids(key, only_events)
    if not ids:
        print('no trials match')
        return
    tid = ids[rank]
    ref, centres = REF_T[key]
    xx, yy, vx, vy, tt, d = trial_dev(key, tid, ref, centres)
    ev = event_of(key, tid)
    an = key.split("|")[0]

    fig = plt.figure(figsize=figsize)
    gs = fig.add_gridspec(5, 2, width_ratios=[1.0, 1.7], hspace=0.22, wspace=0.22)
    axp = fig.add_subplot(gs[:, 0])
    draw_trial_path(axp, xx, yy, tt, ev, colour_by, y_goal=session_time(key)[2])
    axp.set_xlabel('x (vu)')
    axp.set_ylabel('y (vu)')
    axs = []
    for r in range(5):
        axs.append(fig.add_subplot(gs[r, 1], sharex=axs[0] if axs else None))
    draw_trial_timeseries(axs, xx, yy, vx, vy, tt, d,
                          np.interp(yy, centres, ref), ev, thr=EXC_THR_A[an])
    if ev is None:
        verdict = 'spatial only' if (key, tid) in _SPATIAL else 'no event'
        rew = ''
    else:
        verdict = 'CORRECTION  t_out %.2f s  t_back %.2f s' % (ev['t_out'], ev['t_back'])
        rew = '  rewarded' if ev['rewarded'] else '  NO REWARD'
    rank_shown = rank if rank >= 0 else len(ids) + rank
    fig.suptitle('%s  tr%d   rank %d/%d   dev_rms %.2f vu   %.1f s%s   %s'
                 % (key, tid, rank_shown + 1, len(ids), _DEV_RMS[(key, tid)], tt[-1],
                    rew, verdict),
                 x=0.01, ha='left', fontsize=9,
                 color=CRITICAL if ev is not None else INK)
    plt.show()
    return tid

def browse_trials(key=None, figsize=(11.0, 6.4)):
    """Slider version. Needs ipywidgets, whose JS is fetched from a CDN -- if that is blocked
    (VS Code logs 'Widget Error: Failed to access CDN https://unpkg.com'), use show_trial()."""
    import ipywidgets as W
    from IPython.display import display

    all_keys = sorted(TIME_KEYS, key=lambda k: (k.split("|")[0], k.split("|")[1:]))
    key = key or CONTACT_KEYS[-1]
    dd = W.Dropdown(options=all_keys, value=key, description='session',
                    layout=W.Layout(width='330px'))
    colour = W.RadioButtons(options=['time', 'annotation'], value='time', description='colour')
    only = W.Checkbox(value=False, description='only detected events')
    sl = W.IntSlider(value=0, min=0, max=1, description='rank', continuous_update=False,
                     layout=W.Layout(width='420px'))
    prev, nxt = W.Button(description='prev'), W.Button(description='next')
    out = W.Output()

    def refresh(*_a):
        sl.max = max(len(_browse_ids(dd.value, only.value)) - 1, 0)
        sl.value = min(sl.value, sl.max)
        draw()

    def draw(*_a):
        with out:
            out.clear_output(wait=True)
            show_trial(sl.value, dd.value, colour.value, only.value, figsize)

    dd.observe(refresh, 'value'); only.observe(refresh, 'value')
    colour.observe(draw, 'value'); sl.observe(draw, 'value')
    prev.on_click(lambda _b: sl.set_trait('value', max(sl.value - 1, sl.min)))
    nxt.on_click(lambda _b: sl.set_trait('value', min(sl.value + 1, sl.max)))
    refresh()
    display(W.VBox([W.HBox([dd, only]), W.HBox([prev, sl, nxt]), colour, out]))

# Not auto-run: a "Run All" should not build widgets, and this cell deliberately does not switch
# the matplotlib backend -- doing that inside a live kernel, after figures exist, destabilises it.
if 'ipykernel' in sys.modules:
    print('trial browser ready. Widgets need a CDN that may be blocked here, so prefer:')
    print('    show_trial(0)      tidiest trial of the session, by dev_rms')
    print('    show_trial(-1)     worst trial')
    print('    show_trial(-1, colour_by="annotation")')
    print('    show_trial(5, only_events=True)   step through detected corrections')
    print('  session defaults to %s; pass key="animal|date|session" for another.'
          % CONTACT_KEYS[-1])
    print('  slider version (needs working widgets): browse_trials()')
else:
    print('browser defined but not started (no notebook kernel)')

### Event waveforms, cached

Only once the annotation has been checked. Each event is cut out twice — around its onset and
around its peak — resampled onto a fixed grid at the rig's frame rate, and written to disk, so
that every plot below replots from a ~2 MB file instead of re-reading the share.

Each event is **sign-normalised** by `sign(signed_peak)`, so a leftward excursion and a rightward
one both point the same way. Without this the median across events would cancel to roughly zero
and the whole figure would be an artefact of how many animals happened to veer which way.

Samples outside the trial are `NaN`, not zero, and every average below is a `nan`-aware one: a
trial that ends 1.2 s after the excursion closes must not be read as having returned to baseline.

In [ ]:
# --- Step 6: cache the event waveforms -----------------------------------------------------
FS_GRID = 56.0
WIN_ON  = (-3.0, 3.0)      # s around onset
WIN_PK  = (-2.0, 2.0)      # s around peak
CORR_NPZ = os.path.join(CACHE_DIR, 'prelim_cache_corrections.npz')
CORR_CSV = os.path.join(CACHE_DIR, 'prelim_cache_corrections.csv')

T_ON = np.arange(WIN_ON[0], WIN_ON[1] + 1e-9, 1.0 / FS_GRID)
T_PK = np.arange(WIN_PK[0], WIN_PK[1] + 1e-9, 1.0 / FS_GRID)

def _cut(tt, v, t0, grid):
    """Resample v onto t0+grid; NaN wherever the grid falls outside the trial."""
    out = np.interp(t0 + grid, tt, v, left=np.nan, right=np.nan)
    out[(t0 + grid < tt[0]) | (t0 + grid > tt[-1])] = np.nan
    return out

def build_waveforms():
    n = len(exc_t)
    W = {nm: np.full((n, len(T_ON) if nm.endswith('on') else len(T_PK)), np.nan, np.float32)
         for nm in ('dev_on', 'dx_on', 'dy_on', 'dev_pk', 'dx_pk', 'dy_pk')}
    for i, r in enumerate(exc_t.itertuples()):
        ref, centres = REF_T[r.key]
        xx, yy, vx, vy, tt, d = trial_dev(r.key, r.trial, ref, centres)
        sg = 1.0 if r.signed_peak > 0 else -1.0      # sign-normalise: all events veer the same way
        for nm, v, t0, grid in (('dev_on', d * sg, r.t_onset, T_ON),
                                ('dx_on', vx * sg, r.t_onset, T_ON),
                                ('dy_on', vy, r.t_onset, T_ON),
                                ('dev_pk', d * sg, r.t_peak, T_PK),
                                ('dx_pk', vx * sg, r.t_peak, T_PK),
                                ('dy_pk', vy, r.t_peak, T_PK)):
            W[nm][i] = _cut(tt, v, t0, grid)
    return W

# Staleness signature. A row count alone is not enough: correcting the dx sign in the time cache
# left the event COUNT unchanged, so a count-only check silently reused waveforms built from the
# old sign and the phase portrait kept drawing the stale loop. Bind the cache to the cache
# version, the window geometry and the actual event identities instead.
def _wave_sig():
    ids = "".join("%s%d%.4f" % (r.key, r.trial, r.t_onset) for r in exc_t.itertuples())
    import hashlib
    return "%s|%d|%.3f,%.3f|%.3f,%.3f|%s" % (
        TIME_VER, len(exc_t), WIN_ON[0], WIN_ON[1], WIN_PK[0], WIN_PK[1],
        hashlib.md5(ids.encode()).hexdigest()[:12])

SIG = _wave_sig()
WAVE = None
if os.path.exists(CORR_NPZ) and not REBUILD:
    with np.load(CORR_NPZ) as _w:
        if 'sig' in _w.files and str(_w['sig']) == SIG:
            WAVE = {k: _w[k] for k in _w.files if k not in ('T_ON', 'T_PK', 'sig')}
        else:
            print('cached waveforms are stale (signature changed) - rebuilding')
if WAVE is None:
    WAVE = build_waveforms()
    np.savez_compressed(CORR_NPZ, T_ON=T_ON, T_PK=T_PK, sig=np.array(SIG), **WAVE)

exc_t.to_csv(CORR_CSV, index=False)
print("event waveforms: %d events x %d signals, %.1f MB on disk"
      % (len(exc_t), len(WAVE), os.path.getsize(CORR_NPZ) / 1e6))
print("wrote %s" % os.path.basename(CORR_CSV))

A_IDX = {a: np.flatnonzero((exc_t['animal'] == a).to_numpy()) for a in TRAJ_ANIMALS}

def _boot_ci(M, n_boot=400, seed=0):
    """Nan-aware mean with a bootstrap 95% CI over events (rows)."""
    rng = np.random.default_rng(seed)
    m = np.nanmean(M, axis=0)
    idx = rng.integers(0, M.shape[0], size=(n_boot, M.shape[0]))
    bs = np.array([np.nanmean(M[ix], axis=0) for ix in idx])
    return m, np.nanpercentile(bs, 2.5, axis=0), np.nanpercentile(bs, 97.5, axis=0)

### h4c — event-aligned deviation, the core read-out

The direct answer to the question. Every event aligned twice: on its **onset**, which shows how
fast an animal peels away from its route, and on its **peak**, which shows the return and is the
cleaner of the two because the recovery is not smeared by variation in how long the outbound
limb took.

The thin bundle keeps real alpha rather than `flatten_alpha`, because here the overlap *is* the
information — where the lines pile up is where the events agree.

Two things to be careful about when reading this. Both `t_out` and `t_back` are defined by the
threshold crossings, so both scale with `EXC_THR_A`; the threshold is printed on each panel for
that reason. And the median trace flattens toward the window edges partly because trials end —
the traces are `NaN`-padded, so the event count behind each timepoint falls off, which is why the
event count is annotated rather than left implicit.

In [ ]:
# --- h4c: event-aligned deviation traces ---------------------------------------------------
TRIAL_A = 0.16          # real alpha: the bundle overlap is the information

fig, axes = plt.subplots(2, len(TRAJ_ANIMALS), figsize=(3.5 * len(TRAJ_ANIMALS), 6.6),
                         sharex='row', sharey=True)
for j, a in enumerate(TRAJ_ANIMALS):
    ix = A_IDX[a]
    g = exc_t.iloc[ix]
    for i, (grid, M, lab, mark) in enumerate(
            ((T_ON, WAVE['dev_on'][ix], 'time from onset (s)', ('t_out', g['t_out'].median())),
             (T_PK, WAVE['dev_pk'][ix], 'time from peak (s)', ('t_back', g['t_back'].median())))):
        ax = axes[i, j]
        for row in M:
            ax.plot(grid, row, color=BLUE, lw=0.5, alpha=TRIAL_A, zorder=2)
        med = np.nanmedian(M, axis=0)
        q1, q3 = np.nanpercentile(M, 25, axis=0), np.nanpercentile(M, 75, axis=0)
        ax.fill_between(grid, q1, q3, color=flatten_alpha(BLUE, 0.22), lw=0, zorder=3)
        ax.plot(grid, med, color='#104281', lw=2.2, zorder=4)
        ax.axvline(0, color=INK, lw=0.9, zorder=5)
        ax.axhline(EXC_THR_A[a], color=CRITICAL, lw=0.9, ls='--', zorder=5)
        ax.axhline(0, color=BASELINE, lw=0.8, zorder=1)
        ax.axvline(mark[1], color=GOOD, lw=1.2, ls=':', zorder=5)
        ax.set_xlabel(lab, fontsize=8)
        ax.set_title('%s  n=%d events\nmedian %s = %.2f s   thr %.1f vu'
                     % (a, len(ix), mark[0], mark[1], EXC_THR_A[a]), loc='left', fontsize=8.5)
axes[0, 0].set_ylabel('deviation (vu)\naligned on onset')
axes[1, 0].set_ylabel('deviation (vu)\naligned on peak')
fig.suptitle('h4c - how long it takes to leave the route, and how long to get back',
             x=0.01, ha='left', fontsize=11, color=INK)
fig.tight_layout(rect=(0, 0, 1, 0.93))
plt.show()

### h4d — every event, sorted by duration

The median in h4c hides the thing worth knowing: whether corrections come in one flavour or
several. One row per event, sorted by total duration, with the offset time drawn over the top —
that overlaid curve *is* the duration distribution.

A smooth curve means a single continuum of correction speeds. A knee, or two visually distinct
bands, would mean fast flicks and slow excursions are separate behaviours and should not be
pooled into one median.

In [ ]:
# --- h4d: event raster, sorted by duration -------------------------------------------------
_dt = 1.0 / FS_GRID
T_ON_EDGES = np.concatenate([T_ON - 0.5 * _dt, [T_ON[-1] + 0.5 * _dt]])   # pcolormesh wants edges

fig, axes = plt.subplots(1, len(TRAJ_ANIMALS), figsize=(3.6 * len(TRAJ_ANIMALS), 4.8),
                         sharex=True)
axes = np.atleast_1d(axes)
vmax = float(np.nanpercentile(np.abs(WAVE['dev_on']), 99))
for ax, a in zip(axes, TRAJ_ANIMALS):
    ix = A_IDX[a]
    g = exc_t.iloc[ix]
    order = np.argsort(g['t_total'].to_numpy())
    M = np.abs(WAVE['dev_on'][ix][order])                  # (n_events, len(T_ON))
    im = ax.pcolormesh(T_ON_EDGES, np.arange(len(order) + 1), M,
                       cmap=SEQ_CMAP, vmin=0, vmax=vmax, shading='flat', rasterized=True)
    # Clipped in the data, so the handful of long events sit on the right edge and read as
    # "at least this long" rather than running off and rescaling the panel.
    ax.plot(np.clip(g['t_total'].to_numpy()[order], T_ON[0], T_ON[-1]),
            np.arange(len(order)) + 0.5, color=CRITICAL, lw=1.4, zorder=3)
    ax.axvline(0, color=INK, lw=0.9, zorder=3)
    # Clip to the cached window. The duration distribution has a long tail (a handful of events
    # run to 15-37 s, where the animal pauses mid-excursion), and letting the offset line set the
    # limits squeezes the entire raster into an unreadable strip at the left.
    ax.set_xlim(T_ON[0], T_ON[-1])
    n_over = int((g['t_total'] > T_ON[-1]).sum())
    ax.set_xlabel('time from onset (s)', fontsize=8)
    ax.set_title('%s  n=%d events\n%d past the window (max %.0f s)'
                 % (a, len(ix), n_over, g['t_total'].max()), loc='left', fontsize=8)
    ax.grid(False)
axes[0].set_ylabel('event, sorted by duration')
cb = fig.colorbar(im, ax=axes.tolist(), fraction=0.02, pad=0.01)
cb.set_label('|deviation| (vu)', fontsize=8); cb.ax.tick_params(labelsize=7)
fig.suptitle('h4d - the spread of correction timescales; the red line is each event offset',
             x=0.01, ha='left', fontsize=11, color=INK)
plt.show()

### h4e — is correcting slower than erring?

`t_out` against `t_back`, as distributions and as a joint scatter. Three questions, one figure:

* Does getting back on route take longer than leaving it? Compare the two histograms, and read
  the scatter against the unity line.
* Does a bigger excursion cost proportionally more time? That is the colour.
* Is the relationship tight or is duration essentially independent of amplitude? That is the
  spread around the trend.

Both axes are bounded below by the sampling interval and above by the excursion width criterion,
so the ranges are not free — the shape within them is what carries the information.

In [ ]:
# --- h4e: t_out / t_back distributions and their joint structure ---------------------------
_hi = float(np.nanpercentile(exc_t[['t_out', 't_back']].to_numpy().ravel(), 99))
BINS = np.linspace(0, _hi, 26)

fig, axes = plt.subplots(len(TRAJ_ANIMALS), 3, figsize=(10.2, 2.9 * len(TRAJ_ANIMALS)),
                         sharex='col')
for i, a in enumerate(TRAJ_ANIMALS):
    g = exc_t[exc_t['animal'] == a]
    for j, (col, lab) in enumerate(((g['t_out'], 't_out  (onset to peak)'),
                                    (g['t_back'], 't_back  (peak to offset)'))):
        ax = axes[i, j]
        ax.hist(col, bins=BINS, color=flatten_alpha(BLUE, 0.55), edgecolor=SURFACE, lw=0.4)
        ax.axvline(col.median(), color=CRITICAL, lw=1.4)
        ax.set_title('%s   %s   median %.2f s' % (a, lab, col.median()), loc='left', fontsize=8)
        if i == len(TRAJ_ANIMALS) - 1:
            ax.set_xlabel('seconds', fontsize=8)
        if j == 0:
            ax.set_ylabel('events')

    ax = axes[i, 2]
    sc = ax.scatter(g['t_out'], g['t_back'], c=g['peak'], cmap=SEQ_CMAP, s=13,
                    linewidths=0, alpha=0.85)
    lim = (0, _hi)
    ax.plot(lim, lim, color=MUTED, lw=1.0, ls='--', zorder=1)
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_title('%s   slower to correct than to err: %.0f%%'
                 % (a, 100.0 * (g['t_back'] > g['t_out']).mean()), loc='left', fontsize=8)
    ax.set_ylabel('t_back (s)', fontsize=8)
    if i == len(TRAJ_ANIMALS) - 1:
        ax.set_xlabel('t_out (s)', fontsize=8)
    cb = fig.colorbar(sc, ax=ax, fraction=0.045, pad=0.02)
    cb.set_label('peak dev (vu)', fontsize=7); cb.ax.tick_params(labelsize=6)
fig.suptitle('h4e - how long each half of a correction takes, and whether size buys time',
             x=0.01, ha='left', fontsize=11, color=INK)
fig.tight_layout(rect=(0, 0, 1, 0.955))
plt.show()

### h4f — the correction as a dynamical system

Lateral velocity against deviation, averaged over events and coloured by time. Because `dx` is
recorded rather than differentiated, this costs nothing and is the most compact summary of the
correction available.

The out and back limbs separate into a loop, and its shape is the read-out:

* a **wide loop** means the animal is still moving outward at deviations it later corrects from —
  the correction lags the error, which is what a slow feedback controller looks like
* a **narrow loop** hugging a single curve means `dx` is a function of deviation alone: the animal
  corrects at a rate set by how far off it is, with little hysteresis
* where the return limb crosses `dx = 0` is the peak; how steeply it descends afterwards is how
  hard the animal drives itself back

In [ ]:
# --- h4f: phase portrait, lateral velocity against deviation -------------------------------
fig, axes = plt.subplots(1, len(TRAJ_ANIMALS), figsize=(3.6 * len(TRAJ_ANIMALS), 4.2))
axes = np.atleast_1d(axes)
lc_last = None
for ax, a in zip(axes, TRAJ_ANIMALS):
    ix = A_IDX[a]
    mdev = np.nanmean(WAVE['dev_pk'][ix], axis=0)
    mdx = np.nanmean(WAVE['dx_pk'][ix], axis=0)
    ok = np.isfinite(mdev) & np.isfinite(mdx)
    lc_last = colored_line(ax, mdev[ok], mdx[ok], c=T_PK[ok], cmap=TIME_CMAP, lw=2.4,
                           vmin=T_PK[0], vmax=T_PK[-1])
    # direction arrows along the loop
    idx = np.flatnonzero(ok)
    for p in idx[::max(len(idx) // 9, 1)][1:-1]:
        ax.annotate('', xy=(mdev[p + 1], mdx[p + 1]), xytext=(mdev[p], mdx[p]),
                    arrowprops=dict(arrowstyle='-|>', color=INK_2, lw=0.8, alpha=0.8))
    ax.scatter([mdev[ok][0]], [mdx[ok][0]], s=34, facecolor=SURFACE, edgecolor=INK,
               lw=1.0, zorder=6)
    ax.axhline(0, color=BASELINE, lw=0.9)
    ax.axvline(EXC_THR_A[a], color=CRITICAL, lw=0.9, ls='--')
    ax.set_xlabel('deviation (vu)', fontsize=8)
    ax.set_title('%s  n=%d events' % (a, len(ix)), loc='left', fontsize=9)
    ax.grid(True)
axes[0].set_ylabel('lateral velocity dx (vu/s)')
cb = fig.colorbar(lc_last, ax=axes.tolist(), fraction=0.02, pad=0.01)
cb.set_label('time from peak (s)', fontsize=8); cb.ax.tick_params(labelsize=7)
# y > 1 keeps the figure title clear of the per-panel titles. tight_layout() is avoided here
# because the colorbar was placed with ax=<list of axes>, and the two layout engines fight
# over the same space.
fig.suptitle('h4f - the correction as a loop: outward limb above dx=0, return below.  '
             'Open circle marks the window start.',
             x=0.01, y=1.06, ha='left', fontsize=11, color=INK)
plt.show()

### h4g — does the animal slow down to correct?

With view angle locked there is no turn to measure, so forward speed `dy` is the main behavioural
covariate left. Aligned to onset, with `|dx|` stacked below it rather than on a second axis.

If `dy` dips around the peak the correction is costly and the animal trades forward progress for
lateral accuracy. If `dy` is flat the correction is free, folded into ongoing running. The `|dx|`
panel is the control: it *must* rise around the event, since that is what the detector selected
on — if it does not, something is wrong upstream.

**Do not read the `|dx|` peak at t = 0 as a finding.** Onset is *defined* as the moment `|dev|`
crosses the threshold, so events are aligned on a moment when lateral motion is necessarily fast:
the peak at zero is put there by the alignment. It was checked for a discontinuity and there is
none — the mean steps smoothly through the alignment sample (2.49 → 2.31 → 2.19 vu/s for JB1,
against a typical sample-to-sample step of 0.03) — so the decay afterwards is real, and says the
animal is already easing off its lateral push well before the deviation peaks. The same caution
applies to the convergence at t = 0 in h4c. The peak-aligned panels are the ones to trust for
shape, since the peak is found *within* an event rather than being what defines its edge.

In [ ]:
# --- h4g: forward speed and lateral speed, aligned to onset --------------------------------
fig, axes = plt.subplots(2, len(TRAJ_ANIMALS), figsize=(3.5 * len(TRAJ_ANIMALS), 5.4),
                         sharex=True, sharey='row')
for j, a in enumerate(TRAJ_ANIMALS):
    ix = A_IDX[a]
    g = exc_t.iloc[ix]
    t_back_med = g['t_back'].median()
    for i, (M, lab, col) in enumerate(
            ((WAVE['dy_on'][ix], 'forward speed dy (vu/s)', BLUE),
             (np.abs(WAVE['dx_on'][ix]), 'lateral speed |dx| (vu/s)', ORANGE))):
        ax = axes[i, j]
        m, lo, hi = _boot_ci(M)
        ax.fill_between(T_ON, lo, hi, color=flatten_alpha(col, 0.25), lw=0, zorder=2)
        ax.plot(T_ON, m, color=col, lw=2.0, zorder=3)
        ax.axvline(0, color=INK, lw=0.9, zorder=4)
        ax.axvline(g['t_out'].median(), color=CRITICAL, lw=1.0, ls=':', zorder=4)
        ax.axvline(g['t_out'].median() + t_back_med, color=GOOD, lw=1.0, ls=':', zorder=4)
        if i == 0:
            ax.set_title('%s  n=%d events' % (a, len(ix)), loc='left', fontsize=9)
        if j == 0:
            ax.set_ylabel(lab, fontsize=8)
        if i == 1:
            ax.set_xlabel('time from onset (s)', fontsize=8)
fig.suptitle('h4g - is a correction costly? dotted lines mark the median peak and offset',
             x=0.01, ha='left', fontsize=11, color=INK)
fig.tight_layout(rect=(0, 0, 1, 0.93))
plt.show()

### h4h — where on the track do corrections start, and does it matter?

The H3 detector computes `bin_start` and `peak_bin` for every excursion and then never plots
them. This is that locus, against duration.

The marginal histogram answers where along the track deviations begin — the detector excludes the
first and last few bins by construction, so the edges of the range are not evidence. The scatter
answers whether an excursion that starts early, with plenty of track left to recover in, takes
longer to close than one starting near the tower where the animal is running out of room.

In [ ]:
# --- h4h: onset position against correction duration ---------------------------------------
Y_BINS = np.linspace(0, 200, 11)
# Shared, robust y-limit. A single 37 s event otherwise compresses every ordinary correction into
# the bottom tenth of the panel. Off-scale points are counted in each title rather than dropped.
T_CAP = float(np.percentile(exc_t['t_total'], 98))

fig, axes = plt.subplots(2, len(TRAJ_ANIMALS), figsize=(3.5 * len(TRAJ_ANIMALS), 5.2),
                         sharex=True, sharey='row', height_ratios=[1, 2.6])
for j, a in enumerate(TRAJ_ANIMALS):
    g = exc_t[exc_t['animal'] == a]

    axes[0, j].hist(g['y_onset'], bins=Y_BINS, color=flatten_alpha(BLUE, 0.55),
                    edgecolor=SURFACE, lw=0.4)
    n_over = int((g['t_total'] > T_CAP).sum())
    axes[0, j].set_title('%s  n=%d events   %d above %.1f s (max %.0f s)'
                         % (a, len(g), n_over, T_CAP, g['t_total'].max()),
                         loc='left', fontsize=8)
    if j == 0:
        axes[0, j].set_ylabel('events', fontsize=8)

    ax = axes[1, j]
    # Clipped in the data, so off-scale events sit on the top edge and read as "at least this
    # long"; the panel title says how many there are and how far they really go.
    sc = ax.scatter(g['y_onset'], np.minimum(g['t_total'], T_CAP), c=g['peak'],
                    cmap=SEQ_CMAP, s=14, linewidths=0, alpha=0.85)
    y_on = g['y_onset'].to_numpy()
    t_tot = g['t_total'].to_numpy()
    which = np.digitize(y_on, Y_BINS) - 1
    cx, cy = [], []
    for b in range(len(Y_BINS) - 1):
        m = which == b
        if m.sum() >= 5:
            cx.append(0.5 * (Y_BINS[b] + Y_BINS[b + 1]))
            cy.append(float(np.median(t_tot[m])))
    if cx:
        ax.plot(cx, cy, color=CRITICAL, lw=2.0, marker='o', ms=4, zorder=4)
    ax.set_ylim(0, T_CAP)
    ax.set_xlabel('y at onset (vu)  -  tower at 200', fontsize=8)
    if j == 0:
        ax.set_ylabel('event duration t_total (s)', fontsize=8)
    cb = fig.colorbar(sc, ax=ax, fraction=0.045, pad=0.02)
    cb.set_label('peak dev (vu)', fontsize=7); cb.ax.tick_params(labelsize=6)
fig.suptitle('h4h - where corrections begin, and whether starting early buys time\n'
             'red line: median duration per 20 vu bin (bins with >=5 events)',
             x=0.01, ha='left', fontsize=11, color=INK)
fig.tight_layout(rect=(0, 0, 1, 0.92))
plt.show()

In [ ]:
# --- H4 summary table ----------------------------------------------------------------------
h4 = (exc_t.groupby('animal')
      .agg(n_events=('trial', 'size'),
           thr_vu=('animal', lambda s: EXC_THR_A[s.iloc[0]]),
           t_out_med=('t_out', 'median'), t_back_med=('t_back', 'median'),
           t_total_med=('t_total', 'median'),
           peak_med=('peak', 'median'), y_onset_med=('y_onset', 'median'),
           frac_rewarded=('rewarded', 'mean'),
           frac_backtrack=('max_backtrack', lambda s: (s > MAX_BACKTRACK_VU).mean()))
      .round(3))
h4['t_back_over_t_out'] = (h4['t_back_med'] / h4['t_out_med']).round(3)
print(h4.to_string())

H4_CSV = os.path.join(CACHE_DIR, 'prelim_results_h4.csv')
h4.to_csv(H4_CSV)
print('\nwrote %s' % os.path.basename(H4_CSV))

# The duration distribution is heavy-tailed, and the tail is not a second kind of correction --
# it is the animal stopping partway through one. t_total keeps counting while the animal is
# stationary, so those events measure dwell time, not correction speed. Medians are quoted
# everywhere above for this reason; h4d and h4h clip their axes and say how many fall outside.
print('\nduration tail -- events are timed, not gated on the animal still moving:')
for a in TRAJ_ANIMALS:
    g = exc_t[exc_t['animal'] == a]
    print('   %-5s  median %.2f s   90th %.2f s   99th %.2f s   max %.1f s   >5 s: %d of %d'
          % (a, g['t_total'].median(), g['t_total'].quantile(0.90),
             g['t_total'].quantile(0.99), g['t_total'].max(),
             int((g['t_total'] > 5).sum()), len(g)))
print('Check a few of these in the browser: if the animal is stationary the event is a pause,')
print('not a slow correction, and gating on dy would be the fix.')

# Order-of-magnitude check against an independent route. An excursion covers a known stretch of
# track, so its duration and that stretch imply a forward speed, which can be compared with the
# animal's own median speed. A ratio near 1 means the event boundaries are in a sensible place;
# a ratio of 10, or 0.1, would mean they are not.
_sp = ss[ss['animal'].isin(TRAJ_ANIMALS)].groupby('animal')['med_speed_cm_s'].median()
print('\nplausibility check -- implied forward speed during an excursion vs the trial median:')
for a in TRAJ_ANIMALS:
    g = exc_t[exc_t['animal'] == a]
    implied = (g['yspan'].median() * CM_PER_VU) / g['t_total'].median()
    med = float(_sp.get(a, np.nan))
    print('   %-5s  yspan %5.1f vu in %.2f s -> %5.1f cm/s implied   trial median %5.1f cm/s'
          '   ratio %.2f' % (a, g['yspan'].median(), g['t_total'].median(), implied, med,
                             implied / max(med, 1e-6)))
print('\nThese come out around 2x the trial-median speed, which is the right direction:')
print('med_speed_cm_s averages over the whole traverse including slow starts and near-tower')
print('dwelling, whereas an excursion is something the animal does while actually running --')
print('the dy panel of h4g is the direct check on that.')